# Cleaning Pipeline
Make a clean Pipeline version, without ECC, cropping and realigning of older pictures.
Keep the crop for the new pictures, to get rid of everything that is not the satellite photo.

In [ ]:
#############################
# FULL PIPELINE – CLEAN VERSION
# No ECC Alignment, no Water Mask, no FINAL_COMMON_BOX.
# Preprocessing: Apply a fixed “after‐2016” crop then resize to IMG_SIZE.
#############################

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime

import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.applications import EfficientNetV2S, ConvNeXtTiny
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras import Model

#############################
# 1) GLOBAL PARAMETERS
#############################
IMG_SIZE = (224, 224)   # Final model input resolution

TRAIN_IN = "../data/train_new"       # subfolders: sea_ice, no_ice, cloudy
# (Assumed structure for training data: each folder contains images for one class)

#############################
# 2) PREPROCESSING FUNCTIONS
#############################
def fixed_crop(pil_img):
    """
    Applies a fixed crop corresponding to the "after-2016" geometry.
    Crop parameters: left = 110, top = 272, right = (width - 437), bottom = (height - 113).
    """
    w, h = pil_img.size
    left = 110
    top = 272
    right = w - 437
    bottom = h - 113
    return pil_img.crop((left, top, right, bottom))

def process_training_image(fp_str):
    """
    Loads an image from disk, applies the fixed crop,
    and resizes the result to IMG_SIZE.
    """
    pil_img = Image.open(fp_str).convert("RGB")
    cropped = fixed_crop(pil_img)
    final_img = cropped.resize(IMG_SIZE)
    return final_img

#############################
# 3) TF DATASET PIPELINE
#############################
def load_image_for_training(fp, lbl):
    fp_str = fp.numpy().decode("utf-8")
    out_img = process_training_image(fp_str)
    arr = np.array(out_img, dtype=np.float32) / 255.0
    return arr, lbl.numpy()

def build_dataset(filepaths, labels, batch_size=8, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(len(filepaths), reshuffle_each_iteration=True)
    def map_func(fp, lbl):
        arr, lab = tf.py_function(func=load_image_for_training, inp=[fp, lbl],
                                  Tout=[tf.float32, tf.int32])
        arr.set_shape((IMG_SIZE[0], IMG_SIZE[1], 3))
        lab.set_shape(())
        return arr, lab
    ds = ds.map(map_func, num_parallel_calls=tf.data.experimental.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)
    return ds

#############################
# 4) MODEL BUILDING
#############################
def build_classifier(base_name="efficientnetv2", num_classes=2, lr=1e-5, full_fine_tune=True):
    if base_name.lower() == "efficientnetv2":
        base = EfficientNetV2S(weights="imagenet", include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    elif base_name.lower() == "convnext":
        base = ConvNeXtTiny(weights="imagenet", include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    else:
        raise ValueError("Unknown base_name:", base_name)
    base.trainable = full_fine_tune
    x = GlobalAveragePooling2D()(base.output)
    x = Dropout(0.2)(x)
    out = Dense(num_classes, activation="softmax")(x)
    model = Model(base.input, out)
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

#############################
# 5) EVALUATION UTILS
#############################
def evaluate_model_on_test(model, test_ds, class_names):
    all_labels = []
    all_preds = []
    for imgs, lbls in test_ds:
        preds = model.predict(imgs)
        preds_class = np.argmax(preds, axis=1)
        all_labels.extend(lbls.numpy())
        all_preds.extend(preds_class)
    cm = confusion_matrix(all_labels, all_preds)
    print("Confusion Matrix:\n", cm)
    print(classification_report(all_labels, all_preds, target_names=class_names))

def plot_training_history(history, title="Training History"):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history["accuracy"], label="Train Accuracy")
    plt.plot(history.history["val_accuracy"], label="Val Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title + " - Accuracy")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " - Loss")
    plt.legend()
    plt.show()

#############################
# 6) CLASSIFYING NEW IMAGES & VISUALIZATION
#############################
def classify_new_image(image_path, model):
    """
    Loads and processes a single image, feeds it to the model,
    and returns the predicted label index.
    """
    proc_img = process_training_image(image_path)
    arr = np.array(proc_img, dtype=np.float32) / 255.0
    inp = np.expand_dims(arr, axis=0)
    preds = model.predict(inp)
    return np.argmax(preds, axis=1)[0]

def visualize_prediction(image_path, model):
    lbl_idx = classify_new_image(image_path, model)
    proc_img = process_training_image(image_path)
    plt.figure(figsize=(5, 5))
    plt.imshow(proc_img)
    plt.title(f"Predicted Label: {lbl_idx}")
    plt.axis("off")
    plt.show()

#############################
# 7) MAIN ORCHESTRATION (TRAINING)
#############################
def main():
    # --- Build DataFrames for training data ---
    print("=== Building DataFrames from TRAIN_IN ===")
    def build_two_step_data(base_dir):
        subs = ["sea_ice", "no_ice", "cloudy"]
        cpaths, clabels = [], []
        ipaths, ilabels = [], []
        for sf in subs:
            fullp = os.path.join(base_dir, sf)
            for fn in os.listdir(fullp):
                if fn.lower().endswith((".jpg", ".png")):
                    fp = os.path.join(fullp, fn)
                    # For cloud classification: cloudy=1, others=0
                    if sf == "cloudy":
                        cpaths.append(fp)
                        clabels.append(1)
                    else:
                        cpaths.append(fp)
                        clabels.append(0)
                    # For ice classification: sea_ice=0, no_ice=1
                    if sf == "sea_ice":
                        ipaths.append(fp)
                        ilabels.append(0)
                    elif sf == "no_ice":
                        ipaths.append(fp)
                        ilabels.append(1)
        df_cloud = pd.DataFrame({"filepath": cpaths, "label": clabels})
        df_ice = pd.DataFrame({"filepath": ipaths, "label": ilabels})
        return df_cloud, df_ice

    def split_df(df, frac_train=0.8, frac_val=0.1):
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)
        n = len(df)
        n_train = int(frac_train * n)
        n_val = int(frac_val * n)
        df_train = df.iloc[:n_train]
        df_val = df.iloc[n_train:n_train+n_val]
        df_test = df.iloc[n_train+n_val:]
        return df_train, df_val, df_test

    df_cloud, df_ice = build_two_step_data(TRAIN_IN)
    train_cloud, val_cloud, test_cloud = split_df(df_cloud)
    train_ice, val_ice, test_ice = split_df(df_ice)

    # --- Build TensorFlow Datasets ---
    print("=== Building TF Datasets (in memory pipeline) ===")
    def map_func(fp, lbl):
        fp_str = fp.numpy().decode("utf-8")
        img = process_training_image(fp_str)
        arr = np.array(img, dtype=np.float32) / 255.0
        return arr, lbl.numpy()
    def build_ds(filepaths, labels, shuffle=True, batch=8):
        ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
        if shuffle:
            ds = ds.shuffle(len(filepaths), reshuffle_each_iteration=True)
        def _mapper(fp, lab):
            arr, lb = tf.py_function(func=map_func, inp=[fp, lab],
                                     Tout=[tf.float32, tf.int32])
            arr.set_shape((IMG_SIZE[0], IMG_SIZE[1], 3))
            lb.set_shape(())
            return arr, lb
        ds = ds.map(_mapper, num_parallel_calls=tf.data.experimental.AUTOTUNE)
        ds = ds.batch(batch).prefetch(tf.data.experimental.AUTOTUNE)
        return ds
    train_cloud_ds = build_ds(df_cloud["filepath"].tolist(), df_cloud["label"].tolist(), True, 8)
    val_cloud_ds = build_ds(df_cloud["filepath"].tolist(), df_cloud["label"].tolist(), False, 8)
    test_cloud_ds = build_ds(df_cloud["filepath"].tolist(), df_cloud["label"].tolist(), False, 8)
    
    # --- Train Cloud Model ---
    print("=== Train Cloud Model ===")
    cloud_model = build_classifier("efficientnetv2", num_classes=2, lr=1e-5, full_fine_tune=True)
    cloud_checkpoint = ModelCheckpoint("best_cloud_model.keras",
                                       monitor="val_loss",
                                       verbose=1,
                                       save_best_only=True,
                                       mode="min")
    hist_cloud = cloud_model.fit(train_cloud_ds, validation_data=val_cloud_ds, epochs=30,
                                 callbacks=[cloud_checkpoint])
    plot_training_history(hist_cloud, "Cloud Classifier")
    print("Evaluate Cloud Model on test set:")
    cloud_model.evaluate(test_cloud_ds)
    evaluate_model_on_test(cloud_model, test_cloud_ds, ["clear", "cloudy"])
    
    # --- Train Ice Model ---
    print("=== Train Ice Model ===")
    def map_func_ice(fp, lbl):
        fp_str = fp.numpy().decode("utf-8")
        img = process_training_image(fp_str)
        arr = np.array(img, dtype=np.float32) / 255.0
        return arr, lbl.numpy()
    def build_ice_ds(filepaths, labels, shuffle=True, batch=8):
        ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
        if shuffle:
            ds = ds.shuffle(len(filepaths))
        def _mapper(fp, lb):
            ar, l = tf.py_function(func=map_func_ice, inp=[fp, lb],
                                   Tout=[tf.float32, tf.int32])
            ar.set_shape((IMG_SIZE[0], IMG_SIZE[1], 3))
            l.set_shape(())
            return ar, l
        ds = ds.map(_mapper, num_parallel_calls=tf.data.experimental.AUTOTUNE)
        ds = ds.batch(batch).prefetch(tf.data.experimental.AUTOTUNE)
        return ds
    train_ice_ds = build_ice_ds(df_ice["filepath"].tolist(), df_ice["label"].tolist(), True, 8)
    val_ice_ds = build_ice_ds(df_ice["filepath"].tolist(), df_ice["label"].tolist(), False, 8)
    test_ice_ds = build_ice_ds(df_ice["filepath"].tolist(), df_ice["label"].tolist(), False, 8)
    
    ice_model = build_classifier("efficientnetv2", num_classes=2, lr=1e-5, full_fine_tune=True)
    ice_checkpoint = ModelCheckpoint("best_ice_model.keras",
                                     monitor="val_loss",
                                     verbose=1,
                                     save_best_only=True,
                                     mode="min")
    hist_ice = ice_model.fit(train_ice_ds, validation_data=val_ice_ds, epochs=30,
                             callbacks=[ice_checkpoint])
    plot_training_history(hist_ice, "Ice Classifier")
    print("Evaluate Ice Model on test set:")
    ice_model.evaluate(test_ice_ds)
    evaluate_model_on_test(ice_model, test_ice_ds, ["sea_ice", "no_ice"])
    
    print("=== Pipeline Complete ===")

if __name__ == "__main__":
    main()


# Train New Cloud Model for +2016 Images

In [ ]:
#############################
# FULL PIPELINE – CLEAN VERSION
# No ECC Alignment, no Water Mask, no FINAL_COMMON_BOX.
# Preprocessing: Apply a fixed “after‐2016” crop then resize to IMG_SIZE.
#############################

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime

import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.applications import EfficientNetV2S, ConvNeXtTiny
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras import Model

#############################
# 1) GLOBAL PARAMETERS
#############################
IMG_SIZE = (224, 224)   # Final model input resolution

TRAIN_IN = "../data/train_new"       # subfolders: sea_ice, no_ice, cloudy
# (Assumed structure for training data: each folder contains images for one class)

#############################
# 2) PREPROCESSING FUNCTIONS
#############################
def fixed_crop(pil_img):
    """
    Applies a fixed crop corresponding to the "after-2016" geometry.
    Crop parameters: left = 110, top = 272, right = (width - 437), bottom = (height - 113).
    """
    w, h = pil_img.size
    left = 110
    top = 272
    right = w - 437
    bottom = h - 113
    return pil_img.crop((left, top, right, bottom))

def process_training_image(fp_str):
    """
    Loads an image from disk, applies the fixed crop,
    and resizes the result to IMG_SIZE.
    """
    pil_img = Image.open(fp_str).convert("RGB")
    cropped = fixed_crop(pil_img)
    final_img = cropped.resize(IMG_SIZE)
    return final_img

#############################
# 3) TF DATASET PIPELINE
#############################
def load_image_for_training(fp, lbl):
    fp_str = fp.numpy().decode("utf-8")
    out_img = process_training_image(fp_str)
    arr = np.array(out_img, dtype=np.float32) / 255.0
    return arr, lbl.numpy()

def build_dataset(filepaths, labels, batch_size=8, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(len(filepaths), reshuffle_each_iteration=True)
    def map_func(fp, lbl):
        arr, lab = tf.py_function(func=load_image_for_training, inp=[fp, lbl],
                                  Tout=[tf.float32, tf.int32])
        arr.set_shape((IMG_SIZE[0], IMG_SIZE[1], 3))
        lab.set_shape(())
        return arr, lab
    ds = ds.map(map_func, num_parallel_calls=tf.data.experimental.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)
    return ds

#############################
# 4) MODEL BUILDING
#############################
def build_classifier(base_name="efficientnetv2", num_classes=2, lr=1e-5, full_fine_tune=True):
    if base_name.lower() == "efficientnetv2":
        base = EfficientNetV2S(weights="imagenet", include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    elif base_name.lower() == "convnext":
        base = ConvNeXtTiny(weights="imagenet", include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    else:
        raise ValueError("Unknown base_name:", base_name)
    base.trainable = full_fine_tune
    x = GlobalAveragePooling2D()(base.output)
    x = Dropout(0.2)(x)
    out = Dense(num_classes, activation="softmax")(x)
    model = Model(base.input, out)
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

#############################
# 5) EVALUATION UTILS
#############################
def evaluate_model_on_test(model, test_ds, class_names):
    all_labels = []
    all_preds = []
    for imgs, lbls in test_ds:
        preds = model.predict(imgs)
        preds_class = np.argmax(preds, axis=1)
        all_labels.extend(lbls.numpy())
        all_preds.extend(preds_class)
    cm = confusion_matrix(all_labels, all_preds)
    print("Confusion Matrix:\n", cm)
    print(classification_report(all_labels, all_preds, target_names=class_names))

def plot_training_history(history, title="Training History"):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history["accuracy"], label="Train Accuracy")
    plt.plot(history.history["val_accuracy"], label="Val Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title + " - Accuracy")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " - Loss")
    plt.legend()
    plt.show()

#############################
# 6) CLASSIFYING NEW IMAGES & VISUALIZATION
#############################
def classify_new_image(image_path, model):
    """
    Loads and processes a single image, feeds it to the model,
    and returns the predicted label index.
    """
    proc_img = process_training_image(image_path)
    arr = np.array(proc_img, dtype=np.float32) / 255.0
    inp = np.expand_dims(arr, axis=0)
    preds = model.predict(inp)
    return np.argmax(preds, axis=1)[0]

def visualize_prediction(image_path, model):
    lbl_idx = classify_new_image(image_path, model)
    proc_img = process_training_image(image_path)
    plt.figure(figsize=(5, 5))
    plt.imshow(proc_img)
    plt.title(f"Predicted Label: {lbl_idx}")
    plt.axis("off")
    plt.show()

#############################
# 7) MAIN ORCHESTRATION (TRAINING)
#############################
def main():
    # --- Build DataFrames for training data ---
    print("=== Building DataFrames from TRAIN_IN ===")
    def build_two_step_data(base_dir):
        subs = ["sea_ice", "no_ice", "cloudy"]
        cpaths, clabels = [], []
        ipaths, ilabels = [], []
        for sf in subs:
            fullp = os.path.join(base_dir, sf)
            for fn in os.listdir(fullp):
                if fn.lower().endswith((".jpg", ".png")):
                    fp = os.path.join(fullp, fn)
                    # For cloud classification: cloudy=1, others=0
                    if sf == "cloudy":
                        cpaths.append(fp)
                        clabels.append(1)
                    else:
                        cpaths.append(fp)
                        clabels.append(0)
                    # For ice classification: sea_ice=0, no_ice=1
                    if sf == "sea_ice":
                        ipaths.append(fp)
                        ilabels.append(0)
                    elif sf == "no_ice":
                        ipaths.append(fp)
                        ilabels.append(1)
        df_cloud = pd.DataFrame({"filepath": cpaths, "label": clabels})
        df_ice = pd.DataFrame({"filepath": ipaths, "label": ilabels})
        return df_cloud, df_ice

    def split_df(df, frac_train=0.8, frac_val=0.1):
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)
        n = len(df)
        n_train = int(frac_train * n)
        n_val = int(frac_val * n)
        df_train = df.iloc[:n_train]
        df_val = df.iloc[n_train:n_train+n_val]
        df_test = df.iloc[n_train+n_val:]
        return df_train, df_val, df_test

    df_cloud, df_ice = build_two_step_data(TRAIN_IN)
    train_cloud, val_cloud, test_cloud = split_df(df_cloud)
    train_ice, val_ice, test_ice = split_df(df_ice)

    # --- Build TensorFlow Datasets ---
    print("=== Building TF Datasets (in memory pipeline) ===")
    def map_func(fp, lbl):
        fp_str = fp.numpy().decode("utf-8")
        img = process_training_image(fp_str)
        arr = np.array(img, dtype=np.float32) / 255.0
        return arr, lbl.numpy()
    def build_ds(filepaths, labels, shuffle=True, batch=8):
        ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
        if shuffle:
            ds = ds.shuffle(len(filepaths), reshuffle_each_iteration=True)
        def _mapper(fp, lab):
            arr, lb = tf.py_function(func=map_func, inp=[fp, lab],
                                     Tout=[tf.float32, tf.int32])
            arr.set_shape((IMG_SIZE[0], IMG_SIZE[1], 3))
            lb.set_shape(())
            return arr, lb
        ds = ds.map(_mapper, num_parallel_calls=tf.data.experimental.AUTOTUNE)
        ds = ds.batch(batch).prefetch(tf.data.experimental.AUTOTUNE)
        return ds
    train_cloud_ds = build_ds(df_cloud["filepath"].tolist(), df_cloud["label"].tolist(), True, 8)
    val_cloud_ds = build_ds(df_cloud["filepath"].tolist(), df_cloud["label"].tolist(), False, 8)
    test_cloud_ds = build_ds(df_cloud["filepath"].tolist(), df_cloud["label"].tolist(), False, 8)
    
    # --- Train Cloud Model ---
    print("=== Train Cloud Model ===")
    cloud_model = build_classifier("efficientnetv2", num_classes=2, lr=1e-5, full_fine_tune=True)
    cloud_checkpoint = ModelCheckpoint("best_cloud_model.keras",
                                       monitor="val_loss",
                                       verbose=1,
                                       save_best_only=True,
                                       mode="min")
    hist_cloud = cloud_model.fit(train_cloud_ds, validation_data=val_cloud_ds, epochs=30,
                                 callbacks=[cloud_checkpoint])
    plot_training_history(hist_cloud, "Cloud Classifier")
    print("Evaluate Cloud Model on test set:")
    cloud_model.evaluate(test_cloud_ds)
    evaluate_model_on_test(cloud_model, test_cloud_ds, ["clear", "cloudy"])
    
    # --- Train Ice Model ---
    print("=== Train Ice Model ===")
    def map_func_ice(fp, lbl):
        fp_str = fp.numpy().decode("utf-8")
        img = process_training_image(fp_str)
        arr = np.array(img, dtype=np.float32) / 255.0
        return arr, lbl.numpy()
    def build_ice_ds(filepaths, labels, shuffle=True, batch=8):
        ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
        if shuffle:
            ds = ds.shuffle(len(filepaths))
        def _mapper(fp, lb):
            ar, l = tf.py_function(func=map_func_ice, inp=[fp, lb],
                                   Tout=[tf.float32, tf.int32])
            ar.set_shape((IMG_SIZE[0], IMG_SIZE[1], 3))
            l.set_shape(())
            return ar, l
        ds = ds.map(_mapper, num_parallel_calls=tf.data.experimental.AUTOTUNE)
        ds = ds.batch(batch).prefetch(tf.data.experimental.AUTOTUNE)
        return ds
    train_ice_ds = build_ice_ds(df_ice["filepath"].tolist(), df_ice["label"].tolist(), True, 8)
    val_ice_ds = build_ice_ds(df_ice["filepath"].tolist(), df_ice["label"].tolist(), False, 8)
    test_ice_ds = build_ice_ds(df_ice["filepath"].tolist(), df_ice["label"].tolist(), False, 8)
    
    ice_model = build_classifier("efficientnetv2", num_classes=2, lr=1e-5, full_fine_tune=True)
    ice_checkpoint = ModelCheckpoint("best_ice_model.keras",
                                     monitor="val_loss",
                                     verbose=1,
                                     save_best_only=True,
                                     mode="min")
    hist_ice = ice_model.fit(train_ice_ds, validation_data=val_ice_ds, epochs=30,
                             callbacks=[ice_checkpoint])
    plot_training_history(hist_ice, "Ice Classifier")
    print("Evaluate Ice Model on test set:")
    ice_model.evaluate(test_ice_ds)
    evaluate_model_on_test(ice_model, test_ice_ds, ["sea_ice", "no_ice"])
    
    print("=== Pipeline Complete ===")

if __name__ == "__main__":
    main()


### K Means Testing

In [ ]:
import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime

import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.cluster import KMeans

from tensorflow.keras.applications import EfficientNetV2S, ConvNeXtTiny
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras import Model

########################
# GLOBALS
########################
IMG_SIZE = (224, 224)   # Final model input resolution
TRAIN_IN = "../data/train"  # subfolders: sea_ice, no_ice, cloudy
# e.g. "sea_ice/*.jpg", "cloudy/*.jpg"

########################
# 1) Crop + Resize (Same for images + mask)
########################
def fixed_crop(pil_img):
    """
    Applies a fixed crop for "after-2016" geometry.
    left=110, top=272, right=(width-437), bottom=(height-113).
    """
    w, h = pil_img.size
    left   = 110
    top    = 110
    right  = w - 437
    bottom = h - 113
    return pil_img.crop((left, top, right, bottom))

def process_image_consistently(pil_img, is_mask=False):
    """
    1) If already (224,224), assume it’s preprocessed -> do nothing.
    2) Else apply fixed_crop + resize.
    3) If is_mask, use NEAREST for resizing, else BILINEAR (default).
    """
    cropped = fixed_crop(pil_img)
    if is_mask:
        return cropped.resize(IMG_SIZE, Image.NEAREST)
    else:
        return cropped.resize(IMG_SIZE)  # default = BILINEAR

########################
# 2) For training images
########################
def process_training_image(fp_str):
    """
    Loads an image from disk, ensuring it's preprocessed
    exactly once with the fixed crop & resize.
    """
    pil_img = Image.open(fp_str).convert("RGB")
    return process_image_consistently(pil_img, is_mask=False)

########################
# 3) For the mask
########################
def load_land_mask(mask_path):
    """
    Loads a single-channel land mask.  
    Returns a NumPy array with shape (224,224) in {0, 255}.
    """
    pil_mask = Image.open(mask_path).convert("L")
    processed_mask = process_image_consistently(pil_mask, is_mask=True)
    return np.array(processed_mask, dtype=np.uint8)

########################
# 4) K-Means Clustering
########################
def cluster_image_with_mask(image_path, mask_path, k=3):
    """
    - Preprocess the image (crop + resize once).
    - Preprocess the mask (same approach).
    - K-Means on non-land (255) pixels -> returns color-coded cluster map.
    """
    # (A) Preprocess the image
    pil_img = Image.open(image_path).convert("RGB")
    pil_img = process_image_consistently(pil_img, is_mask=False)
    img_arr = np.array(pil_img, dtype=np.float32) / 255.0

    # (B) Preprocess the mask
    land_mask = load_land_mask(mask_path)

    # Flatten + apply K-Means
    H, W, _ = img_arr.shape
    flat_img  = img_arr.reshape(-1, 3)
    flat_mask = land_mask.reshape(-1)
    non_land_idxs = np.where(flat_mask == 255)[0]
    non_land_pixels = flat_img[non_land_idxs]

    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(non_land_pixels)
    cluster_labels = kmeans.labels_

    # Reconstruct
    full_labels = np.full((H*W,), fill_value=-1, dtype=np.int32)
    full_labels[non_land_idxs] = cluster_labels

    # Color-code result
    palette = [
        (255, 0,   0),   # cluster 0 => red
        (0,   255, 0),   # cluster 1 => green
        (0,   0,   255), # cluster 2 => blue
        (255, 255, 0),
        (255, 0,   255)
    ]
    land_color = (0, 0, 0)
    output_map = np.zeros((H, W, 3), dtype=np.uint8)
    for i in range(H*W):
        label = full_labels[i]
        if label == -1:
            output_map[i//W, i%W] = land_color
        else:
            color = palette[label % len(palette)]
            output_map[i//W, i%W] = color
    return output_map, full_labels.reshape(H, W), img_arr, land_mask

def visualize_clustered_image(clustered_map):
    """Display the K-Means color-coded map."""
    plt.figure(figsize=(5,5))
    plt.imshow(clustered_map)
    plt.title("Unsupervised Clustering Map (K-Means)")
    plt.axis("off")
    plt.show()

########################
# 5) Optional: Overlay or Compare
########################
def overlay_cluster_result(img_arr, cluster_map):
    """
    Overlays the cluster_map (RGB) with some transparency on top of the original image (img_arr).
    This helps you see if the black (land) region lines up with the coastline properly.
    """
    H, W, _ = img_arr.shape
    # Convert arrays to PIL
    base_img = Image.fromarray((img_arr*255).astype(np.uint8)).convert("RGBA")
    overlay_img = Image.fromarray(cluster_map).convert("RGBA")

    # Create a blend => e.g. 70% base, 30% cluster
    blended = Image.blend(base_img, overlay_img, alpha=0.3)
    return blended

########################
# 6) Example MAIN
########################
def main():
    # Here, we skip the entire training pipeline for brevity,
    # focusing on the K-Means + mask alignment check.

    print("=== Demo: Clustering a single actual image with a new mask ===")
    example_image_path = "../data/satellite/aqua/2025/20250219_AQUA.jpg"
    example_mask_path  = "../data/satellite/aqua/mask/mask_AQUA_new.jpg"

    if os.path.exists(example_image_path) and os.path.exists(example_mask_path):
        cluster_map, label_map, img_arr, land_mask = cluster_image_with_mask(example_image_path,
                                                                             example_mask_path,
                                                                             k=3)
        # Show the cluster result by itself
        visualize_clustered_image(cluster_map)

        # Optionally overlay to see alignment
        blended = overlay_cluster_result(img_arr, cluster_map)
        plt.figure(figsize=(7,7))
        plt.imshow(blended)
        plt.title("Overlay: Original Image + K-Means Result (semi-transparent)")
        plt.axis("off")
        plt.show()
    else:
        print("Skipping demo because files not found:", example_image_path, example_mask_path)

    print("=== Done ===")


if __name__ == "__main__":
    main()


# Create K-Means Time Series out of all Photos to K-Means Time Series.csv

In [ ]:
import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime

from sklearn.cluster import KMeans

########################
# 1) GLOBAL PARAMETERS
########################
IMG_SIZE = (224, 224)   # final image size after crop + resize
K = 3                   # we want 3 clusters => water, ice, weak ice
DATA_ROOT = "../data/satellite/aqua"  # e.g. subfolders 2016, 2017, ...
LAND_MASK_PATH = "../data/satellite/aqua/mask/mask_AQUA_new.jpg"  # single global mask
DO_CROP = True          # whether to apply the “after-2016” geometry crop

# For color-coding
COLOR_WATER    = (255,   0,   0)  # red
COLOR_ICE      = (0,   255,   0)  # green
COLOR_WEAK_ICE = (0,     0, 255)  # blue
COLOR_LAND     = (0,     0,   0)  # black

########################
# 2) HELPER FUNCTIONS
########################

def fixed_crop(pil_img):
    """
    Example of a fixed crop for “after-2016 geometry.”
    left=110, top=110, right=(width-437), bottom=(height-113)
    Adjust these numbers if your actual geometry differs.
    """
    w, h = pil_img.size
    left   = 110
    top    = 110
    right  = w - 437
    bottom = h - 113
    return pil_img.crop((left, top, right, bottom))

def process_image_consistently(fp, is_mask=False, do_crop=True, target_size=(224,224)):
    """
    Loads an image (RGB for normal images, L for mask).
    If do_crop=True, applies fixed_crop. Then resizes to target_size.
    
    - For normal images (is_mask=False):
        * Convert to RGB
        * Resize with default bilinear
        * Return float32 array in [0..1]
    - For masks (is_mask=True):
        * Convert to L
        * Resize with NEAREST
        * Return uint8 array in {0,255}
    """
    pil_img = Image.open(fp)
    if is_mask:
        pil_img = pil_img.convert("L")
    else:
        pil_img = pil_img.convert("RGB")

    if do_crop:
        pil_img = fixed_crop(pil_img)

    if is_mask:
        # Use NEAREST for masks
        pil_img = pil_img.resize(target_size, Image.NEAREST)
        arr = np.array(pil_img, dtype=np.uint8)  # mask => 0 or 255
    else:
        # For normal images, use bilinear (default)
        pil_img = pil_img.resize(target_size)
        arr = np.array(pil_img, dtype=np.float32) / 255.0

    return arr

def parse_date_from_filename(filename):
    """
    Example parse: '20160215_AQUA.jpg' => datetime(2016,2,15).
    Adjust as needed for your naming scheme.
    """
    base = os.path.basename(filename)
    # e.g. '20160215_AQUA.jpg'
    # strip extension, split by '_'
    name_part = os.path.splitext(base)[0]  # => '20160215_AQUA'
    tokens = name_part.split('_')
    if len(tokens) < 1:
        return None
    date_str = tokens[0]  # e.g. '20160215'
    if len(date_str) == 8:
        try:
            yyyy = int(date_str[0:4])
            mm   = int(date_str[4:6])
            dd   = int(date_str[6:8])
            return datetime(yyyy, mm, dd)
        except:
            return None
    return None

########################
# 3) K-MEANS UTIL
########################

def assign_clusters_to_labels(img_pixels, cluster_labels):
    """
    We have cluster_labels in {0, 1, 2} from K-Means,
    but we need to map them to water=0, ice=1, weak ice=2 based on brightness:

      darkest => water (0)
      brightest => ice (1)
      middle => weak ice (2)
    """
    unique_clust = list(np.unique(cluster_labels))
    if len(unique_clust) < 3:
        pass  # partial fallback

    brightnesses = []
    for c in unique_clust:
        c_pixels = img_pixels[cluster_labels == c]
        if len(c_pixels) > 0:
            avg_bright = c_pixels.mean(axis=1).mean()
        else:
            avg_bright = 0
        brightnesses.append((c, avg_bright))

    brightnesses.sort(key=lambda x: x[1])  # ascending

    if len(brightnesses) == 3:
        darkest_id   = brightnesses[0][0]
        middle_id    = brightnesses[1][0]
        brightest_id = brightnesses[2][0]
        cluster_to_label = {
            darkest_id:   0,  # water
            middle_id:    2,  # weak ice
            brightest_id: 1   # ice
        }
    elif len(brightnesses) == 2:
        darkest_id   = brightnesses[0][0]
        brightest_id = brightnesses[1][0]
        cluster_to_label = {
            darkest_id:   0,
            brightest_id: 1
        }
    else:
        c_id = brightnesses[0][0] if len(brightnesses)>0 else 0
        cluster_to_label = { c_id: 0 }

    new_labels = np.full_like(cluster_labels, fill_value=-1)
    for c_id, final_lbl in cluster_to_label.items():
        mask = (cluster_labels == c_id)
        new_labels[mask] = final_lbl
    return new_labels


def cluster_one_image(fp, land_mask_arr):
    """
    1) Load normal image with process_image_consistently(is_mask=False).
    2) Flatten non-land pixels (where mask=255).
    3) K-Means => 3 clusters => re-labeled to water=0, ice=1, weak ice=2.
    4) Build color-coded map, compute fractions => return (seg_map, fractions).
    """
    # Load normal image
    img_arr = process_image_consistently(fp, is_mask=False, do_crop=DO_CROP, target_size=IMG_SIZE)
    H, W, _ = img_arr.shape
    flat_img = img_arr.reshape(-1, 3)

    flat_mask = land_mask_arr.reshape(-1)  # shape: (H*W,)
    non_land_idx = np.where(flat_mask == 255)[0]
    non_land_pixels = flat_img[non_land_idx]

    if len(non_land_pixels) == 0:
        seg_map = np.zeros((H, W, 3), dtype=np.uint8)
        return seg_map, {"water":0.0, "ice":0.0, "weak_ice":0.0}

    # K-Means
    kmeans = KMeans(n_clusters=K, random_state=42)
    kmeans.fit(non_land_pixels)
    cluster_labels = kmeans.labels_

    # Assign water(0)/ice(1)/weak_ice(2)
    assigned_labels = assign_clusters_to_labels(non_land_pixels, cluster_labels)

    # Reconstruct full label map
    full_labels = np.full((H*W,), fill_value=-1, dtype=np.int32)
    full_labels[non_land_idx] = assigned_labels

    # Convert to color-coded
    seg_map = np.zeros((H, W, 3), dtype=np.uint8)
    for i in range(H*W):
        lbl = full_labels[i]
        if lbl == -1:
            seg_map[i // W, i % W] = COLOR_LAND
        elif lbl == 0:
            seg_map[i // W, i % W] = COLOR_WATER
        elif lbl == 1:
            seg_map[i // W, i % W] = COLOR_ICE
        elif lbl == 2:
            seg_map[i // W, i % W] = COLOR_WEAK_ICE

    total_nonland = len(non_land_pixels)
    frac_water    = np.sum(assigned_labels == 0) / total_nonland
    frac_ice      = np.sum(assigned_labels == 1) / total_nonland
    frac_weakice  = np.sum(assigned_labels == 2) / total_nonland
    fractions = {
        "water": frac_water,
        "ice": frac_ice,
        "weak_ice": frac_weakice
    }
    return seg_map, fractions

def visualize_result(original_arr, seg_map):
    """
    Display original (cropped/resized) on left, segmentation on right.
    original_arr: (H, W, 3) float32 in [0..1]
    seg_map: (H, W, 3) uint8
    """
    pil_original = Image.fromarray((original_arr*255).astype(np.uint8))
    pil_segment  = Image.fromarray(seg_map)

    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1)
    plt.imshow(pil_original)
    plt.title("Original (cropped/resized)")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(pil_segment)
    plt.title("K-Means: Water/Ice/WeakIce")
    plt.axis("off")

    plt.show()

########################
# 4) MAIN PIPELINE
########################
def main():
    # (A) Load the land mask once => use the same approach
    #     so the mask is cropped/resized identically to the image.
    land_mask_arr = process_image_consistently(LAND_MASK_PATH, is_mask=True,
                                               do_crop=DO_CROP, target_size=IMG_SIZE)

    rows = []
    # (B) Iterate years 2016..2025
    for year in range(2016, 2026):
        folder = os.path.join(DATA_ROOT, str(year))
        if not os.path.isdir(folder):
            print(f"Skipping {folder}, not found.")
            continue

        images = glob.glob(os.path.join(folder, "*.*"))
        images = [img for img in images if img.lower().endswith((".jpg", ".png"))]
        images.sort()

        for img_fp in images:
            dt = parse_date_from_filename(img_fp)
            seg_map, fractions = cluster_one_image(img_fp, land_mask_arr)
            row = {
                "filepath": img_fp,
                "date": dt,
                "frac_water": fractions["water"],
                "frac_ice": fractions["ice"],
                "frac_weak_ice": fractions["weak_ice"]
            }
            rows.append(row)

    df = pd.DataFrame(rows)
    df = df.sort_values("date").reset_index(drop=True)
    print("Sample results:\n", df.head(10))

    # (C) Optionally visualize one example
    if len(df) > 0:
        example_fp = df.loc[0,"filepath"]
        print("Visualizing first example =>", example_fp)
        # load original (cropped/resized)
        original_arr = process_image_consistently(example_fp, is_mask=False,
                                                  do_crop=DO_CROP, target_size=IMG_SIZE)
        seg_map, _ = cluster_one_image(example_fp, land_mask_arr)
        visualize_result(original_arr, seg_map)

    out_csv = "kmeans_time_series.csv"
    df.to_csv(out_csv, index=False)
    print("Saved time series data to", out_csv)

if __name__ == "__main__":
    main()


# Create Reference Dataset from Partial K-Means

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime

from sklearn.cluster import KMeans

########################
# GLOBALS
########################
IMG_SIZE = (224, 224)
K = 3  # We want 3 clusters => e.g. water=0, ice=1, weak_ice=2
LAND_MASK_PATH = "../data/satellite/aqua/mask/mask_AQUA.jpg"
DO_CROP = True

########################
# 1) Crop + Resize
########################
def fixed_crop(pil_img):
    """
    Applies a fixed crop for “after-2016 geometry.”
    Adjust to match your actual geometry.

    Example: left=110, top=272, right=(width-437), bottom=(height-113)
    """
    w, h = pil_img.size
    left   = 110
    top    = 272
    right  = w - 437
    bottom = h - 113
    return pil_img.crop((left, top, right, bottom))

def process_image_consistently(fp, is_mask=False, do_crop=True, target_size=(224,224)):
    """
    If is_mask=False, loads an RGB image, applies fixed crop (if do_crop=True), then resizes (bilinear).
      returns float32 array in [0..1]
    If is_mask=True, loads L (grayscale), same crop, then resizes with NEAREST to preserve {0,255}.
      returns uint8 array
    """
    pil_img = Image.open(fp)
    if is_mask:
        pil_img = pil_img.convert("L")
    else:
        pil_img = pil_img.convert("RGB")

    if do_crop:
        pil_img = fixed_crop(pil_img)

    if is_mask:
        pil_img = pil_img.resize(target_size, Image.NEAREST)
        arr = np.array(pil_img, dtype=np.uint8)  # mask => 0 or 255
    else:
        pil_img = pil_img.resize(target_size)    # default = bilinear
        arr = np.array(pil_img, dtype=np.float32) / 255.0
    return arr

########################
# 2) For the reference data creation
########################
def cluster_reference_images(ref_folder, out_csv="ref_pixels.csv"):
    """
    1) Loop over images in ref_folder.
    2) For each, do K-Means -> get cluster_labels for non-land pixels.
    3) Save a cluster map so you can identify which cluster=water, etc.
    4) We'll store (R,G,B, cluster_id, image_id, x, y) in a CSV for possible re-labeling.
    """
    # Load the land mask once, same approach as everything else
    land_mask = process_image_consistently(LAND_MASK_PATH, is_mask=True, do_crop=DO_CROP, target_size=IMG_SIZE)

    rows = []
    image_list = [os.path.join(ref_folder, f) for f in os.listdir(ref_folder)
                  if f.lower().endswith(('.jpg', '.png'))]
    image_list.sort()

    for i, fp in enumerate(image_list):
        # Step A: load + preprocess the normal image
        arr = process_image_consistently(fp, is_mask=False, do_crop=DO_CROP, target_size=IMG_SIZE)
        H, W, _ = arr.shape

        # Flatten + select non-land
        land_flat = land_mask.reshape(-1)  # shape = (H*W,)
        img_flat  = arr.reshape(-1, 3)
        nonland_idx = np.where(land_flat == 255)[0]

        if len(nonland_idx)==0:
            print("No non-land pixels in", fp)
            continue

        # Step B: K-Means
        kmeans = KMeans(n_clusters=K, random_state=42)
        kmeans.fit(img_flat[nonland_idx])
        cluster_labels = kmeans.labels_  # shape = (N,)

        # Step C: Build a full-label map for visualization
        full_labels = np.full((H*W,), -1, dtype=int)
        full_labels[nonland_idx] = cluster_labels

        # e.g. color palette for clusters [0=red,1=green,2=blue], land=black
        palette = [(255,0,0),(0,255,0),(0,0,255)]
        seg_map = np.zeros((H,W,3), dtype=np.uint8)
        for idx_px in range(H*W):
            lbl = full_labels[idx_px]
            if lbl == -1:
                seg_map[idx_px//W, idx_px%W] = (0,0,0)  # land
            else:
                seg_map[idx_px//W, idx_px%W] = palette[lbl % len(palette)]

        # Step D: Save or show
        out_seg = fp + "_clusters.png"
        Image.fromarray(seg_map).save(out_seg)
        print(f"[{fp}] K-Means done, cluster map => {out_seg}")

        # Step E: Store for later re-label => (R, G, B, cluster_id, image_id, x, y)
        for idx_px in range(H*W):
            lbl = full_labels[idx_px]
            if lbl != -1:
                r,g,b = arr[idx_px//W, idx_px%W]
                rows.append({
                    "filepath": fp,
                    "image_index": i,
                    "x": idx_px%W,
                    "y": idx_px//W,
                    "r": r, "g": g, "b": b,
                    "cluster_id": lbl
                })

    # Save CSV
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"Saved reference pixel data to {out_csv}.\n"
          "Next step: manually assign each cluster_id => semantic label in a separate column, or partial edits.")


def main():
    ref_folder = "../data/train_new_kmeans"  # a folder of ~5–10 relatively clear images
    cluster_reference_images(ref_folder, out_csv="ref_pixels_label.csv")

if __name__ == "__main__":
    main()


# Train a KNN on the Verified Pixels & Apply to New Images

In [ ]:
import sys
sys.setrecursionlimit(20000)
import os
import numpy as np
import pandas as pd
from PIL import Image
import joblib  # for saving the trained model
from sklearn.neighbors import KNeighborsClassifier
import pickle

############## CONFIG ##############
IMG_SIZE = (224, 224)
LAND_MASK_PATH = "../data/satellite/aqua/mask/mask_AQUA.jpg"
TRAINED_MODEL_PATH = "water_ice_knn.pkl"
REF_CSV = "ref_pixels.csv"  # This CSV has: filepath, image_index, x, y, r, g, b, cluster_id
####################################

def fixed_crop(pil_img):
    w, h = pil_img.size
    left = 110
    top = 272
    right = w - 437
    bottom = h - 113
    return pil_img.crop((left, top, right, bottom))

def load_and_preprocess(fp):
    img = Image.open(fp).convert("RGB")
    img = fixed_crop(img)
    img = img.resize(IMG_SIZE)
    arr = np.array(img, dtype=np.float32) / 255.0
    return arr

def train_knn(ref_csv=REF_CSV, out_model=TRAINED_MODEL_PATH):
    """
    Reads ref_pixels.csv which contains unsupervised cluster data with columns:
      filepath, image_index, x, y, r, g, b, cluster_id
      
    For each pixel, brightness is computed as (r+g+b)/3.
    Then, for each cluster_id, the average brightness is determined.
    The cluster with the lowest brightness is assigned "water",
    the middle is "thin sea ice", and the highest is "sea ice".
    
    These semantic labels are then mapped to numeric values:
       water       -> 0
       thin sea ice-> 1
       sea ice     -> 2
       
    Finally, a KNeighborsClassifier is trained using (r,g,b) as features.
    """
    df = pd.read_csv(ref_csv)
    # Compute brightness (assumes r, g, b are in [0,1])
    df['brightness'] = (df['r'] + df['g'] + df['b']) / 3.0

    # Group by cluster_id and compute average brightness
    group_df = df.groupby('cluster_id')['brightness'].mean().reset_index()
    group_df = group_df.sort_values('brightness')
    
    # Create mapping from cluster_id to semantic label based on brightness order.
    mapping = {}
    num_clusters = len(group_df)
    if num_clusters >= 3:
        mapping[group_df.iloc[0]['cluster_id']] = "water"
        mapping[group_df.iloc[1]['cluster_id']] = "thin sea ice"
        mapping[group_df.iloc[2]['cluster_id']] = "sea ice"
    elif num_clusters == 2:
        mapping[group_df.iloc[0]['cluster_id']] = "water"
        mapping[group_df.iloc[1]['cluster_id']] = "sea ice"
    else:
        for _, row in group_df.iterrows():
            mapping[row['cluster_id']] = "water"
    
    print("Mapping from cluster_id to semantic label:", mapping)
    
    # Add new 'label' column based on the mapping
    df['label'] = df['cluster_id'].map(mapping)
    
    # Define the numeric mapping for training.
    label_map = {"water": 0, "thin sea ice": 1, "sea ice": 2}
    # Retain only rows with valid labels.
    df = df[df["label"].isin(label_map.keys())]
    
    X = df[["r", "g", "b"]].values
    y = df["label"].map(label_map).values
    
    print("Training data size:", X.shape)
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X, y)
    print("KNN trained. Saving model to:", out_model)
    joblib.dump(knn, out_model, protocol=pickle.HIGHEST_PROTOCOL, compress=0)


def load_knn(model_path=TRAINED_MODEL_PATH):
    return joblib.load(model_path)

def load_land_mask(mask_path, size=(224, 224)):
    pmask = Image.open(mask_path).convert("L")
    pmask = pmask.resize(size)
    arr = np.array(pmask, dtype=np.uint8)
    return arr

def classify_image(img_fp, knn, land_mask, color_map=None):
    """
    1) Loads and preprocesses the image.
    2) Flattens non-land pixels (where land_mask equals 255).
    3) Applies the trained KNN to label each non-land pixel:
         0 = water, 1 = thin sea ice, 2 = sea ice.
    4) Constructs a color segmentation map using the default or provided color_map.
       Default color mapping:
         -1 (land)        -> Black (0, 0, 0)
          0 (water)       -> Red   (255, 0, 0)
          1 (thin sea ice) -> Blue  (0, 0, 255)
          2 (sea ice)     -> Green (0, 255, 0)
    5) Computes the fraction of each class among non-land pixels.
    
    Returns:
      seg_map: (H,W,3) color segmentation image (uint8).
      fractions: dictionary of fractions.
    """
    arr = load_and_preprocess(img_fp)
    H, W, _ = arr.shape
    flat_arr = arr.reshape(-1, 3)

    mask_flat = land_mask.reshape(-1)
    nonland_idx = np.where(mask_flat == 255)[0]

    # Initialize with -1 (land)
    labels_full = np.full((H * W,), fill_value=-1, dtype=int)

    if len(nonland_idx) > 0:
        pred = knn.predict(flat_arr[nonland_idx])
        labels_full[nonland_idx] = pred

    if color_map is None:
        # Default color mapping based on our desired labels.
        color_map = {
            -1: (0, 0, 0),       # Land: Black
             0: (255, 0, 0),     # Water: Red
             1: (0, 0, 255),     # Thin sea ice: Blue
             2: (0, 255, 0)      # Sea ice: Green
        }
    seg_map = np.zeros((H, W, 3), dtype=np.uint8)
    for i in range(H * W):
        lbl = labels_full[i]
        seg_map[i // W, i % W] = color_map.get(lbl, (0, 0, 0))

    # Compute fractions over non-land pixels
    total_nonland = len(nonland_idx)
    frac_water = np.sum(labels_full[nonland_idx] == 0) / float(total_nonland) if total_nonland > 0 else 0
    frac_thin  = np.sum(labels_full[nonland_idx] == 1) / float(total_nonland) if total_nonland > 0 else 0
    frac_sea   = np.sum(labels_full[nonland_idx] == 2) / float(total_nonland) if total_nonland > 0 else 0
    fractions = {
        "water": frac_water,
        "thin sea ice": frac_thin,
        "sea ice": frac_sea
    }
    return seg_map, fractions

def main():
    # 1) Train the KNN using your reference pixels.
    # The CSV "ref_pixels.csv" (in the same directory) contains unsupervised clustering outputs.
    # We automatically assign semantic labels based on brightness.
    train_knn(ref_csv=REF_CSV, out_model=TRAINED_MODEL_PATH)

    # 2) Load the trained KNN model and the land mask.
    knn = load_knn(TRAINED_MODEL_PATH)
    land_mask = load_land_mask(LAND_MASK_PATH, IMG_SIZE)

    # 3) Process and classify new Aqua satellite images from 2025.
    data_folder = "../data/satellite/aqua/2024"
    images = [os.path.join(data_folder, f) for f in os.listdir(data_folder)
              if f.lower().endswith(('.jpg', '.png'))]

    results = []
    for fp in images:
        seg_map, fracs = classify_image(fp, knn, land_mask)
        out_seg = fp + "_classified.png"
        Image.fromarray(seg_map).save(out_seg)
        print(f"Saved segmentation => {out_seg}, fractions={fracs}")

        results.append({
            "filepath": fp,
            "frac_water": fracs["water"],
            "frac_thin_sea_ice": fracs["thin sea ice"],
            "frac_sea_ice": fracs["sea ice"]
        })

    df = pd.DataFrame(results)
    df.to_csv("knn_results_2024.csv", index=False)
    print("Saved knn_results_2024.csv with water/ice fractions per image.")

if __name__ == "__main__":
    main()


# Use first Cloud Model, then KNN on all Images

In [ ]:
#!/usr/bin/env python3
"""
Fast Cloud+Ice/Water pipeline with a 70% black‐pixel filter up front.
"""

import sys
sys.setrecursionlimit(25000)

import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
import joblib
import pickle
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime
from sklearn.neighbors import KNeighborsClassifier

#############################
# CONFIGURATION
#############################
IMG_SIZE = (224, 224)
LAND_MASK_PATH = "../data/satellite/aqua/mask/mask_AQUA_new.jpg"
TRAINED_KNN_MODEL_PATH = "water_ice_knn.pkl"
REF_CSV = "ref_pixels.csv"  # CSV with unsupervised clustering outputs
CLOUD_MODEL_PATH = "backup_best_cloud_model.keras"  # Pretrained cloud model
SATELLITE_ROOT = "../data/satellite/aqua"  # Each subfolder is a year (e.g., 2016..2025)
PDF_OUTPUT = "all_results2.pdf"
CLOUD_THRESHOLD = 0.5  # Probability threshold for cloud detection
DO_CROP = True         # Whether to apply the fixed crop
BLACK_THRESHOLD = 0.10 # Skip if >10% of pixels are pure black

#############################
# 1) Crop + Resize
#############################
def fixed_crop(pil_img):
    w, h = pil_img.size
    left, top = 110, 272
    right, bottom = w - 437, h - 113
    if right < left or bottom < top:
        print(f"ERROR: Image size {w}x{h} is too small for the fixed crop.")
        pil_img.show()
        raise ValueError(f"Invalid crop for size {w}x{h}")
    return pil_img.crop((left, top, right, bottom))

def process_image_consistently(fp, is_mask=False, do_crop=True, target_size=(224,224)):
    pil_img = Image.open(fp)
    pil_img = pil_img.convert("L") if is_mask else pil_img.convert("RGB")
    if do_crop:
        pil_img = fixed_crop(pil_img)
    if is_mask:
        pil_img = pil_img.resize(target_size, Image.NEAREST)
        return np.array(pil_img, dtype=np.uint8)
    else:
        pil_img = pil_img.resize(target_size, Image.BILINEAR)
        return np.array(pil_img, dtype=np.float32) / 255.0

#############################
# 2) Date Parsing
#############################
def extract_date_from_filename(fp):
    base = os.path.basename(fp)
    try:
        return datetime.strptime(base[:8], "%Y%m%d")
    except:
        return None

#############################
# 3) KNN MODEL (water/thin_ice/sea_ice)
#############################
def train_knn(ref_csv=REF_CSV, out_model=TRAINED_KNN_MODEL_PATH):
    df = pd.read_csv(ref_csv)
    df['brightness'] = (df[['r','g','b']].sum(axis=1)) / 3.0
    gm = df.groupby('cluster_id')['brightness'].mean().reset_index().sort_values('brightness')
    mapping = {}
    if len(gm) >= 3:
        mapping[gm.iloc[0].cluster_id] = "water"
        mapping[gm.iloc[1].cluster_id] = "thin sea ice"
        mapping[gm.iloc[2].cluster_id] = "sea ice"
    elif len(gm)==2:
        mapping[gm.iloc[0].cluster_id] = "water"
        mapping[gm.iloc[1].cluster_id] = "sea ice"
    else:
        mapping = {cid:"water" for cid in gm.cluster_id}
    df['label'] = df['cluster_id'].map(mapping)
    lm = {"water":0,"thin sea ice":1,"sea ice":2}
    df = df[df['label'].isin(lm)]
    X = df[['r','g','b']].values; y = df['label'].map(lm).values
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X, y)
    joblib.dump(knn, out_model, protocol=pickle.HIGHEST_PROTOCOL, compress=0)

def load_knn(model_path=TRAINED_KNN_MODEL_PATH):
    return joblib.load(model_path)

#############################
# 4) LAND MASK
#############################
def load_land_mask(mask_path, target_size=(224,224)):
    return process_image_consistently(mask_path, is_mask=True,
                                      do_crop=DO_CROP, target_size=target_size)

#############################
# 5) CLASSIFY IMAGES with KNN
#############################
def classify_image(img_fp, knn, land_mask, color_map=None):
    arr = process_image_consistently(img_fp, is_mask=False,
                                     do_crop=DO_CROP, target_size=IMG_SIZE)
    H, W, _ = arr.shape
    flat = arr.reshape(-1,3)
    mask_flat = land_mask.reshape(-1)
    nonland = np.where(mask_flat==255)[0]
    labels = np.full((H*W,), -1, dtype=int)
    if len(nonland)>0:
        labels[nonland] = knn.predict(flat[nonland])
    cmap = color_map or {
        -1:(0,0,0),0:(255,0,0),1:(0,0,255),2:(0,255,0)
    }
    seg = np.zeros((H,W,3), dtype=np.uint8)
    for idx,val in enumerate(labels):
        seg[idx//W, idx%W] = cmap[val]
    total = len(nonland)
    fracs = {
        "water": (labels[nonland]==0).sum()/total if total else 0,
        "thin sea ice": (labels[nonland]==1).sum()/total if total else 0,
        "sea ice": (labels[nonland]==2).sum()/total if total else 0,
    }
    return seg, fracs

#############################
# 6) CLOUD MODEL
#############################
def is_clear_image(img_fp, cloud_model, threshold=CLOUD_THRESHOLD):
    arr = process_image_consistently(img_fp, is_mask=False,
                                     do_crop=DO_CROP, target_size=IMG_SIZE)
    prob = cloud_model.predict(np.expand_dims(arr,0))[0][0]
    return prob < threshold, prob

#############################
# 7) GENERATE PDF
#############################
def generate_pdf(results_df, pdf_filename):
    results_df['date'] = pd.to_datetime(results_df['date'])
    results_df.sort_values('date', inplace=True)
    with PdfPages(pdf_filename) as pdf:
        # time series
        clear_df = results_df[results_df['cloud_status']=="clear"]
        plt.figure(figsize=(10,6))
        plt.plot(clear_df['date'], clear_df['frac_water'], 'r.-', label='Water')
        plt.plot(clear_df['date'], clear_df['frac_thin sea ice'], 'b.-', label='Thin Sea Ice')
        plt.plot(clear_df['date'], clear_df['frac_sea ice'], 'g.-', label='Sea Ice')
        plt.xlabel("Date"); plt.ylabel("Fraction")
        plt.title("Water/Ice Fractions (Clear Images)"); plt.legend()
        plt.tight_layout(); pdf.savefig(); plt.close()

        # cloud prob scatter
        plt.figure(figsize=(10,6))
        plt.scatter(results_df['date'], results_df['cloud_prob'],
                    c=results_df['cloud_status'].map({"clear":"black","cloudy":"gray"}),
                    alpha=0.7)
        plt.xlabel("Date"); plt.ylabel("Cloud Probability")
        plt.title("Cloud Probability Over Time")
        plt.tight_layout(); pdf.savefig(); plt.close()

        # example seg pages
        sample = clear_df.sample(min(4,len(clear_df)), random_state=42)
        for _,row in sample.iterrows():
            seg_fp = row['filepath']+"_classified.png"
            if os.path.exists(seg_fp):
                img = Image.open(seg_fp)
                plt.figure(figsize=(6,6))
                plt.imshow(img)
                plt.title(f"{row['date'].date()}  W={row['frac_water']:.2f}  Thin={row['frac_thin sea ice']:.2f}  Sea={row['frac_sea ice']:.2f}")
                plt.axis("off"); pdf.savefig(); plt.close()

#############################
# 8) MAIN
#############################
def main():
    print("Loading cloud model…")
    cloud_model = tf.keras.models.load_model(CLOUD_MODEL_PATH)

    if not os.path.exists(TRAINED_KNN_MODEL_PATH):
        print("Training KNN…")
        train_knn()
    knn = load_knn()

    land_mask = load_land_mask(LAND_MASK_PATH)

    all_results = []
    year_dirs = sorted(d for d in os.listdir(SATELLITE_ROOT)
                       if os.path.isdir(os.path.join(SATELLITE_ROOT, d)))
    for year in year_dirs:
        folder = os.path.join(SATELLITE_ROOT, year)
        print("Year folder:", folder)
        files = sorted(f for f in glob.glob(os.path.join(folder, "*.*"))
                       if f.lower().endswith((".jpg",".png"))
                       and "_classified" not in f.lower())

        for fp in files:
            # ——— BLACK‐PIXEL FILTER ———
            img_arr = process_image_consistently(fp, is_mask=False,
                                                 do_crop=DO_CROP,
                                                 target_size=IMG_SIZE)
            black_mask = np.all(img_arr==0, axis=2)
            black_frac = black_mask.sum() / black_mask.size
            if black_frac > BLACK_THRESHOLD:
                print(f"   [SKIP] {os.path.basename(fp)} is {black_frac:.1%} black.")
                continue

            date = extract_date_from_filename(fp)
            clear_flag, cloud_prob = is_clear_image(fp, cloud_model)
            if clear_flag:
                seg_map, fracs = classify_image(fp, knn, land_mask)
                out_seg = fp + "_classified.png"
                Image.fromarray(seg_map).save(out_seg)
                status = "clear"
            else:
                fracs = {"water":None,"thin sea ice":None,"sea ice":None}
                status = "cloudy"

            all_results.append({
                "filepath": fp,
                "date": date,
                "cloud_status": status,
                "cloud_prob": cloud_prob,
                "frac_water": fracs["water"],
                "frac_thin sea ice": fracs["thin sea ice"],
                "frac_sea ice": fracs["sea ice"]
            })

    df = pd.DataFrame(all_results).sort_values("date", na_position="last")
    df.to_csv("all_results2.csv", index=False)
    print("Saved all_results.csv")
    generate_pdf(df, PDF_OUTPUT)
    print(f"Saved PDF: {PDF_OUTPUT}")

if __name__ == "__main__":
    main()


# EDA from all_results.csv

In [ ]:
#!/usr/bin/env python3
"""
Best‑practice EDA for Aqua sea‑ice results
-----------------------------------------
• Reads all_results.csv (same folder by default)
• Cleans & engineers columns
• Generates four publication‑ready plots:
    1) Stacked‑area surface composition (7‑day rolling mean)
    2) Cloud‑status share (%)
    3) Cloud probability trend (7‑day rolling mean)
    4) Seasonal climatology by day‑of‑year (with month labels)
"""

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------------
DATA_PATH      = Path("all_results2.csv")           # adjust if needed
ROLLING_WINDOW = 7                                 # days
FRAC_COLS      = ["frac_water", "frac_thin sea ice", "frac_sea ice"]

# ------------------------------------------------------------------
# 1. Load & tidy
# ------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)

# ------------------------------------------------------------------
# 2. Rolling means for smoother trends
# ------------------------------------------------------------------
rolling = (
    df.set_index("date")[FRAC_COLS]
      .rolling(window=ROLLING_WINDOW, min_periods=1, center=True)
      .mean()
      .fillna(0)
)
cloud_roll = (
    df.set_index("date")["cloud_prob"]
      .rolling(window=ROLLING_WINDOW, min_periods=1, center=True)
      .mean()
)

# ------------------------------------------------------------------
# 3. Plot styles
# ------------------------------------------------------------------
sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

# ------------------------------------------------------------------
# 4. FIGURE 1 – Stacked‑area surface composition
# ------------------------------------------------------------------
plt.figure(figsize=(14, 6))
plt.stackplot(
    rolling.index,
    rolling["frac_water"],
    rolling["frac_thin sea ice"],
    rolling["frac_sea ice"],
    labels=["Water", "Thin sea ice", "Sea ice"],
    alpha=0.85
)
plt.title(f"Surface composition ({ROLLING_WINDOW}‑day rolling mean)")
plt.ylabel("Fraction")
plt.xlabel("Date")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 5. FIGURE 2 – Cloud‑status share (%)
# ------------------------------------------------------------------
cloud_percent = (
    df["cloud_status"]
      .value_counts(normalize=True, dropna=False)
      .sort_index()
) * 100

plt.figure(figsize=(6, 4))
bars = plt.bar(cloud_percent.index.astype(str), cloud_percent.values)
for bar, pct in zip(bars, cloud_percent.values):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        pct + 1,
        f"{pct:.1f}%",
        ha="center",
        va="bottom"
    )
plt.title("Cloud‑status share (%)")
plt.ylabel("Percent of images")
plt.ylim(0, cloud_percent.max() * 1.15)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 6. FIGURE 3 – Cloud probability trend
# ------------------------------------------------------------------
plt.figure(figsize=(14, 4))
plt.plot(cloud_roll.index, cloud_roll.values)
plt.title(f"Cloud probability ({ROLLING_WINDOW}‑day rolling mean)")
plt.ylabel("Probability")
plt.xlabel("Date")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 7. FIGURE 4 – Seasonal climatology (multi‑year average with month labels)
# ------------------------------------------------------------------
# compute day‑of‑year climatology
df["doy"] = df["date"].dt.dayofyear
clim = (
    df.groupby("doy")[FRAC_COLS]
      .mean()
      .rolling(window=ROLLING_WINDOW, min_periods=1, center=True)
      .mean()
)

plt.figure(figsize=(12, 6))
plt.plot(clim.index, clim["frac_water"],        label="Water")
plt.plot(clim.index, clim["frac_thin sea ice"], label="Thin sea ice")
plt.plot(clim.index, clim["frac_sea ice"],      label="Sea ice")
plt.title("Seasonal cycle (multi‑year average)")
plt.xlabel("Month")
plt.ylabel("Mean fraction")

# add month ticks at start of each month
month_starts = pd.date_range("2000-01-01", "2000-12-01", freq="MS")
xticks = month_starts.dayofyear
xticklabels = month_starts.strftime("%b")
plt.xticks(xticks, xticklabels)

plt.xlim(1, 366)
plt.legend()
plt.tight_layout()
plt.show()


# Pipeline with Self built U-Net Approach

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.neighbors import KNeighborsClassifier
import joblib
import pickle

########################
# GLOBAL PARAMETERS
########################
IMG_SIZE = (224, 224)
DO_CROP = True  # whether to apply fixed_crop
DATA_CLOUD_DIR = "../data/cloud_filter"  # the folder with 20 images & 20 masks: e.g. 20160612_AQUA.jpg + 20160612_AQUA_mask.jpg
U_NET_OUTPUT = "cloud_unet.keras"  # Where we save the trained U-Net
EPOCHS_UNET = 15  # Adjust as needed
BATCH_SIZE_UNET = 4  # small batch if you only have 20 images
CLOUD_THRESHOLD_COVERAGE = 0.2  # If fewer than 20% of the pixels are non-cloud, maybe skip the day?

# For the rest of your pipeline:
LAND_MASK_PATH = "../data/satellite/aqua/mask/mask_AQUA_new.jpg"
TRAINED_KNN_MODEL_PATH = "water_ice_knn.pkl"
REF_CSV = "ref_pixels.csv"
SATELLITE_ROOT = "../data/satellite/aqua"
PDF_OUTPUT = "all_results.pdf"

########################
# 1) FIXED CROP
########################
def fixed_crop(pil_img):
    w, h = pil_img.size
    left   = 110
    top    = 272
    right  = w - 437
    bottom = h - 113
    if right < left or bottom < top:
        print(f"ERROR: Image {w}x{h} too small for fixed crop.")
        pil_img.show()
        raise ValueError(f"Invalid crop box => left={left}, top={top}, right={right}, bottom={bottom}")
    return pil_img.crop((left, top, right, bottom))

########################
# 2) UNIVERSAL PREPROCESS
########################
def process_image_consistently(fp, is_mask=False, do_crop=True, target_size=(224,224)):
    """
    - If is_mask=True => convert to L, apply NEAREST resizing, produce uint8 in {0,255}
    - If is_mask=False => convert to RGB, bilinear resizing, produce float32 in [0..1]
    """
    pil_img = Image.open(fp)
    if is_mask:
        pil_img = pil_img.convert("L")
    else:
        pil_img = pil_img.convert("RGB")

    if do_crop:
        pil_img = fixed_crop(pil_img)

    if is_mask:
        pil_img = pil_img.resize(target_size, Image.NEAREST)
        arr = np.array(pil_img, dtype=np.uint8)  # in {0,255}
    else:
        pil_img = pil_img.resize(target_size)
        arr = np.array(pil_img, dtype=np.float32)/255.0
    return arr

########################
# 3) BUILD A SIMPLE U-NET
########################
def build_unet(input_shape=(224,224,3)):
    """
    A simplified U-Net for binary segmentation (cloud=1, background=0).
    For real production, consider deeper unet, batchnorm, etc.
    """
    inputs = layers.Input(shape=input_shape)

    # Downsample
    c1 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2,2))(c1)

    c2 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2,2))(c2)

    # Bottleneck
    c3 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(c3)

    # Upsample
    u4 = layers.UpSampling2D((2,2))(c3)
    u4 = layers.concatenate([u4, c2], axis=-1)
    c4 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(u4)
    c4 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(c4)

    u5 = layers.UpSampling2D((2,2))(c4)
    u5 = layers.concatenate([u5, c1], axis=-1)
    c5 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(u5)
    c5 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c5)

    # final => 1 channel => sigmoid
    outputs = layers.Conv2D(1, (1,1), activation='sigmoid')(c5)
    model = models.Model(inputs=[inputs], outputs=[outputs])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


########################
# 4) LOADING TRAINING DATA FOR CLOUD U-NET
########################
def load_cloud_data(data_dir, do_crop=True, target_size=(224,224)):
    """
    We expect pairs of images:
      "20160612_AQUA.jpg" and "20160612_AQUA_mask.jpg"
    We gather them in X (images) and Y (masks).
    """
    all_jpgs = sorted([f for f in os.listdir(data_dir)
                       if f.lower().endswith(".jpg") and "_mask" not in f.lower()])
    X_list, Y_list = [], []

    for img_name in all_jpgs:
        base_name = os.path.splitext(img_name)[0]  # e.g. "20160612_AQUA"
        mask_name = base_name + "_mask.jpg"
        img_fp  = os.path.join(data_dir, img_name)
        mask_fp = os.path.join(data_dir, mask_name)
        if not os.path.exists(mask_fp):
            print(f"Warning: no mask found for {img_fp} => {mask_fp}, skipping.")
            continue

        # load image => shape (224,224,3)
        img_arr  = process_image_consistently(img_fp, is_mask=False,
                                              do_crop=do_crop, target_size=target_size)
        # load mask => shape (224,224), in {0,255}, we convert to {0,1} float
        mask_arr = process_image_consistently(mask_fp, is_mask=True,
                                              do_crop=do_crop, target_size=target_size)
        mask_arr = (mask_arr>127).astype(np.float32)  # now in {0,1}

        if img_arr.shape != (224,224,3):
            print(f"Skipping {img_fp}, shape mismatch => {img_arr.shape}")
            continue
        if mask_arr.shape != (224,224):
            print(f"Skipping {mask_fp}, shape mismatch => {mask_arr.shape}")
            continue

        X_list.append(img_arr)
        # expand mask to (224,224,1)
        Y_list.append(mask_arr[...,None])

    X = np.array(X_list, dtype=np.float32)
    Y = np.array(Y_list, dtype=np.float32)
    return X, Y

########################
# 5) TRAIN THE U-NET
########################
def train_unet(data_dir=DATA_CLOUD_DIR, epochs=EPOCHS_UNET, batch_size=BATCH_SIZE_UNET):
    """
    Loads the 20 images + masks from data_dir, trains the U-Net for cloud detection,
    then saves "cloud_unet.keras".
    """
    X, Y = load_cloud_data(data_dir, do_crop=DO_CROP, target_size=IMG_SIZE)
    print("Data shapes:", X.shape, Y.shape)  # e.g. (N,224,224,3), (N,224,224,1)
    unet = build_unet(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    history = unet.fit(X, Y, validation_split=0.2,
                       epochs=epochs, batch_size=batch_size)
    unet.save("cloud_unet.keras")
    print("Saved cloud U-Net to cloud_unet.keras")
    return unet, history

########################
# 6) PREDICT CLOUDS (Pixel-Level)
########################
def predict_cloud_mask(unet_model, img_fp, do_crop=True):
    """
    1) Preprocess the normal image => shape=(224,224,3) in [0..1]
    2) unet_model.predict => shape=(1,224,224,1)
    3) threshold => shape=(224,224) in {0,1}, 1=cloud
    Returns cloud_mask with shape=(224,224).
    """
    arr = process_image_consistently(img_fp, is_mask=False,
                                     do_crop=do_crop, target_size=IMG_SIZE)
    inp = np.expand_dims(arr, axis=0)
    pred = unet_model.predict(inp)[0]  # shape=(224,224,1)
    cloud_mask = (pred[...,0]>0.5).astype(np.uint8)  # in {0,1}
    return cloud_mask

########################
# 7) COMBINE WITH LAND MASK + KNN
########################
def classify_image_partial_cloud(img_fp, unet_model, knn_model, land_mask, cloud_coverage_thr=CLOUD_THRESHOLD_COVERAGE):
    """
    Steps:
     1) get arr => shape=(224,224,3)
     2) get cloud_mask => shape=(224,224) in {0=clear,1=cloud}
     3) combine with land_mask => shape=(224,224) in {0=land,255=non-land}
     4) only classify where land=255 and cloud=0
     5) measure coverage => if coverage is too small, skip or label as 'too cloudy'.
    """
    arr = process_image_consistently(img_fp, is_mask=False, do_crop=DO_CROP, target_size=IMG_SIZE)
    H, W, _ = arr.shape

    # Cloud detection
    c_mask = predict_cloud_mask(unet_model, img_fp, do_crop=DO_CROP)
    # Land
    mask_flat = land_mask.reshape(-1)    # {0=land, 255=non-land}
    cloud_flat= c_mask.reshape(-1)       # {0=clear,1=cloud}

    # We want to classify pixels that are land_mask=255 (non-land) and cloud=0
    valid_idx = []
    for i in range(H*W):
        if mask_flat[i]==255 and cloud_flat[i]==0:
            valid_idx.append(i)
    valid_idx = np.array(valid_idx, dtype=int)
    coverage_fraction = len(valid_idx)/(H*W)
    if coverage_fraction < cloud_coverage_thr:
        # e.g. skip or label as "too cloudy"
        seg_map = np.zeros((H,W,3), dtype=np.uint8)
        fractions = {"water":None,"thin sea ice":None,"sea ice":None}
        return seg_map, fractions, coverage_fraction

    # KNN classification => shape=(224*224,3) => only valid_idx
    flat_arr = arr.reshape(-1,3)
    labels_full = np.full((H*W,), fill_value=-1, dtype=int)
    pred = knn_model.predict(flat_arr[valid_idx])
    labels_full[valid_idx] = pred

    # color map: -1 => black, 0 => red, 1 => blue, 2 => green
    c_map = { -1:(0,0,0), 0:(255,0,0), 1:(0,0,255), 2:(0,255,0) }
    seg_map = np.zeros((H,W,3), dtype=np.uint8)
    for i in range(H*W):
        seg_map[i//W, i%W] = c_map.get(labels_full[i], (0,0,0))

    # compute fractions among valid_idx
    n_valid = len(valid_idx)
    frac_water = np.sum(labels_full[valid_idx]==0)/n_valid
    frac_thin  = np.sum(labels_full[valid_idx]==1)/n_valid
    frac_sea   = np.sum(labels_full[valid_idx]==2)/n_valid
    fractions = {
        "water": frac_water,
        "thin sea ice": frac_thin,
        "sea ice": frac_sea
    }
    return seg_map, fractions, coverage_fraction

########################
# 8) MAIN ORCHESTRATION
########################
def main():
    # Step A) Train or load the U-Net for cloud detection
    if not os.path.exists("cloud_unet.keras"):
        print("Training U-Net for cloud detection on data/cloud folder (20 images + 20 masks).")
        unet_model, hist = train_unet(data_dir=DATA_CLOUD_DIR, epochs=100, batch_size=4)
    else:
        print("U-Net model found => skipping training.")
        unet_model = tf.keras.models.load_model("cloud_unet.keras")

    # Step B) Train or load the KNN for water/thin_ice/sea_ice
    if not os.path.exists(TRAINED_KNN_MODEL_PATH):
        print("KNN not found => training from ref CSV =>", REF_CSV)
        train_knn(ref_csv=REF_CSV, out_model=TRAINED_KNN_MODEL_PATH)
    else:
        print("KNN found => skipping training.")
    knn_model = joblib.load(TRAINED_KNN_MODEL_PATH)

    # Step C) Load land mask => same approach => is_mask=True
    land_mask = process_image_consistently(LAND_MASK_PATH, is_mask=True, do_crop=DO_CROP, target_size=IMG_SIZE)

    # Step D) For each year in SATELLITE_ROOT, classify partially
    results = []
    year_folders = [os.path.join(SATELLITE_ROOT, f) for f in os.listdir(SATELLITE_ROOT)
                    if os.path.isdir(os.path.join(SATELLITE_ROOT, f))]
    year_folders.sort()

    for folder in year_folders:
        print("Processing folder =>", folder)
        # find images
        images = glob.glob(os.path.join(folder, "*.*"))
        images = [im for im in images if im.lower().endswith((".jpg",".png"))
                  and "_classified" not in im.lower()]
        images.sort()

        for img_fp in images:
            dt = extract_date_from_filename(img_fp)
            seg_map, fracs, coverage = classify_image_partial_cloud(
                img_fp, unet_model, knn_model, land_mask,
                cloud_coverage_thr=0.2  # e.g. require at least 20% is cloud-free + land-free
            )
            # If coverage is None => we skip? Actually we returned coverage always
            out_seg = img_fp + "_classified.png"
            if fracs["water"] is not None:
                # means coverage >= threshold
                Image.fromarray(seg_map).save(out_seg)
                print(f"[{img_fp}] partial coverage => {coverage*100:.1f}%, fractions => {fracs}")
            else:
                print(f"[{img_fp}] too cloudy => coverage={coverage*100:.1f}% => skip storing classified.")
                out_seg = None

            row = {
                "filepath": img_fp,
                "date": dt,
                "coverage": coverage,
                "frac_water": fracs["water"],
                "frac_thin sea ice": fracs["thin sea ice"],
                "frac_sea ice": fracs["sea ice"]
            }
            results.append(row)

    df = pd.DataFrame(results).sort_values("date")
    out_csv = "partial_cloud_results.csv"
    df.to_csv(out_csv, index=False)
    print("Saved partial cloud results to", out_csv)

    print("=== Done ===")


if __name__ == "__main__":
    main()


# Train U-Net (PyTorch) with our 20 Photos + Mask

In [ ]:
import os
import random
import glob
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim
import torchvision.transforms as T
import matplotlib.pyplot as plt

import segmentation_models_pytorch as smp

# We define a small script-level config
DATA_DIR = "../data/cloud_filter"
MODEL_OUT = "cloud_unet_rgb.pt"
BATCH_SIZE = 2
EPOCHS = 100
VAL_SPLIT = 0.2
IMG_SIZE = (224, 224)  # You can also try (512,512) if you want bigger networks


##############################
# 1) DATASET
##############################
class CloudDataset(Dataset):
    """
    Expects pairs:  e.g. "20160612_AQUA.jpg" and "20160612_AQUA_mask.jpg"
    We'll do basic augmentation (random flips, color jitter) to help with only 20 images.
    White=cloud, black=clear => we convert to {0,1}.
    """
    def __init__(self, image_files, transforms=None):
        self.image_files = image_files
        self.transforms = transforms

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_fp = self.image_files[idx]
        base = os.path.splitext(img_fp)[0]  # e.g. 20160612_AQUA
        mask_fp = base + "_mask.jpg"
        
        # Load the RGB
        img_pil = Image.open(img_fp).convert("RGB")
        # Load the mask
        mask_pil = Image.open(mask_fp).convert("L")  # L => single channel

        # Resize to e.g. 224x224, or 512x512
        img_pil = img_pil.resize(IMG_SIZE, resample=Image.BILINEAR)
        mask_pil= mask_pil.resize(IMG_SIZE, resample=Image.NEAREST)

        # Convert to Tensor
        img = T.ToTensor()(img_pil)   # shape=(3,H,W) in [0..1]
        mask = T.ToTensor()(mask_pil) # shape=(1,H,W) in [0..1]

        # Make mask binary: >0.5 => 1
        mask = (mask>0.5).float()

        # optionally do augmentations
        if self.transforms:
            # This transforms expects img+mask cat or do separate approach
            # We'll do a simple approach: pass them separately to transforms that have the same random seed
            # or we can do e.g. albumentations or kornia. For simplicity:
            augmented = self.transforms(image=img, mask=mask)
            img = augmented["image"]
            mask= augmented["mask"]

        return img, mask

##############################
# 2) BUILD THE DATASET & SPLIT
##############################
def build_datasets(data_dir=DATA_DIR):
    # gather all *jpg that are not _mask
    image_files = []
    for fn in os.listdir(data_dir):
        if fn.lower().endswith(".jpg") and "_mask" not in fn.lower():
            fp = os.path.join(data_dir, fn)
            image_files.append(fp)
    image_files.sort()
    print(f"Found {len(image_files)} images in {data_dir} (RGB + mask).")

    # We'll do a random shuffle
    random.shuffle(image_files)

    # We'll do an 80/20 split
    n = len(image_files)
    n_val = int(VAL_SPLIT * n)
    val_files = image_files[:n_val]
    train_files= image_files[n_val:]

    train_ds = CloudDataset(train_files, transforms=None)
    val_ds   = CloudDataset(val_files,   transforms=None)
    return train_ds, val_ds

##############################
# 3) BUILD THE MODEL
##############################
def build_unet(num_classes=1):
    """
    We'll do a SMP Unet with a smallish backbone for a small dataset.
    We'll do a single output channel => binary segmentation => use BCE or Dice.
    """
    model = smp.Unet(
        encoder_name="resnet34",      # or "mobilenet_v2"
        encoder_weights="imagenet",   # use pretrained ImageNet
        in_channels=3,
        classes=num_classes,          # 1 => single-channel output
        activation=None              # We'll handle activation with BCE or smp losses
    )
    return model

##############################
# 4) TRAINING LOOP (basic PyTorch)
##############################
def train_model(model, train_ds, val_ds, epochs=EPOCHS, lr=1e-4):
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    # We use a typical BCEWithLogitsLoss, or you can do smp.utils.losses.DiceLoss
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    device = torch.device("mps" if torch.mps.is_available() else "cpu")
    model.to(device)

    best_val_loss = float("inf")
    for ep in range(epochs):
        model.train()
        train_loss = 0.0
        for imgs, masks in train_loader:
            imgs = imgs.to(device)
            masks= masks.to(device)

            optimizer.zero_grad()
            logits = model(imgs)  # shape=(B,1,H,W)
            loss = criterion(logits, masks)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * imgs.size(0)

        train_loss /= len(train_loader.dataset)

        # Evaluate on val
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs = imgs.to(device)
                masks= masks.to(device)
                logits= model(imgs)
                loss = criterion(logits, masks)
                val_loss += loss.item() * imgs.size(0)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch [{ep+1}/{epochs}] - train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
        # Save best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODEL_OUT)
            print("    [*] best model saved.")

    print(f"Training complete. Best val_loss={best_val_loss:.4f}")
    # load best model
    model.load_state_dict(torch.load(MODEL_OUT))
    return model

##############################
# MAIN
##############################
def main():
    # 1) Build dataset
    train_ds, val_ds = build_datasets(DATA_DIR)

    # 2) Build model
    model = build_unet(num_classes=1)

    # 3) Train
    model = train_model(model, train_ds, val_ds, epochs=EPOCHS, lr=1e-4)

    # 4) Test on a random example
    if len(val_ds)>0:
        sample_img, sample_mask = val_ds[0]
        # do a forward pass
        device = torch.device("mps" if torch.mps.is_available() else "cpu")
        model.to(device)
        model.eval()
        with torch.no_grad():
            inp = sample_img.unsqueeze(0).to(device)
            logits= model(inp)
            prob  = torch.sigmoid(logits)[0,0].cpu().numpy()
        pred_mask = (prob>0.5).astype(np.uint8)

        # visualize
        fig, ax = plt.subplots(1,3, figsize=(12,5))
        ax[0].imshow(sample_img.permute(1,2,0).numpy())  # (H,W,3)
        ax[0].set_title("Original (val sample)")

        ax[1].imshow(sample_mask[0].numpy(), cmap="gray")
        ax[1].set_title("Ground truth")

        ax[2].imshow(pred_mask, cmap="gray")
        ax[2].set_title("Predicted mask")
        plt.show()

    print("\nDone training. The model is saved to =>", MODEL_OUT)
    print("You can load and run 'predict_cloud_mask_rgb(model, <img_fp>)' on new images.")


if __name__=="__main__":
    main()


In [ ]:
import sys
sys.setrecursionlimit(25000)

import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image
from datetime import datetime

import torch
import torchvision.transforms as T
import segmentation_models_pytorch as smp
from sklearn.neighbors import KNeighborsClassifier
import joblib

############################
# 0) MPS or fallback device
############################
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("[INFO] Using MPS device on Apple Silicon.")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("[INFO] Using CUDA device.")
else:
    device = torch.device("cpu")
    print("[INFO] Using CPU device (no MPS or CUDA).")

############################
# 1) GLOBAL CONFIG
############################
CLOUD_MODEL_PATH  = "cloud_unet_rgb.pt"   # your self-trained cloud detection model
TRAINED_KNN_MODEL = "water_ice_knn.pkl"
LAND_MASK_PATH    = "../data/satellite/aqua/mask/mask_AQUA_new.jpg"
SATELLITE_ROOT    = "../data/satellite/aqua"

IMG_SIZE          = (224,224)
CLOUD_THRESHOLD   = 0.2            # skip partial coverage if <20% valid
BLACK_THRESHOLD   = 0.2            # skip images if >20% of pixels are near black
OUTPUT_PDF        = "cloud_pipeline_inspection.pdf"
OUTPUT_CSV        = "partial_cloud_results.csv"
MAX_VISUALS       = 100
VISUALS_FOLDER    = "visuals"
os.makedirs(VISUALS_FOLDER, exist_ok=True)

############################
# 2) BUILD + LOAD CLOUD MODEL
############################
def build_unet_rgb():
    """
    Rebuild the same architecture used during your training:
      - e.g. a 3-channel input U-Net with 1-class (binary).
    """
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=3,
        classes=1
    )
    return model

def load_cloud_model(model_path, device):
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Cloud model not found => {model_path}")
    model = build_unet_rgb()
    state_dict = torch.load(model_path, map_location="cpu")
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

############################
# 3) FIXED CROP + PREPROCESS
############################
def fixed_crop(pil_img):
    w, h = pil_img.size
    left   = 110
    top    = 272
    right  = w - 437
    bottom = h - 113
    if right<left or bottom<top:
        print(f"[ERROR] image {w}x{h} too small for fixed crop.")
        pil_img.show()
        raise ValueError("Invalid crop")
    return pil_img.crop((left, top, right, bottom))

def process_image(fp, do_crop=True, size=(224,224)):
    pil_img = Image.open(fp).convert("RGB")
    if do_crop:
        pil_img = fixed_crop(pil_img)
    pil_img = pil_img.resize(size, Image.BILINEAR)
    arr = np.array(pil_img, dtype=np.float32)/255.0  # shape=(H,W,3)
    return arr

############################
# 4) FILTER OUT "TOO BLACK"
############################
def is_too_black(arr, black_thresh=0.7):
    """
    arr => shape=(H,W,3), in [0..1].
    We'll define "near black" as, say, R+G+B < 0.01 or some small number.
    Then if fraction of near-black pixels > black_thresh => skip.
    """
    # We define near-black if sum(R,G,B)< 0.01 e.g.
    # adjust if needed
    near_black = (arr.sum(axis=2) < 0.01)
    frac_black = near_black.mean()
    return (frac_black >= black_thresh), frac_black

############################
# 5) PREDICT CLOUD BINARY
############################
def predict_cloud_mask(model, img_fp, device, do_crop=True):
    arr = process_image(img_fp, do_crop=do_crop, size=IMG_SIZE)
    # check blackness
    skip, black_ratio= is_too_black(arr, black_thresh=BLACK_THRESHOLD)
    if skip:
        # return None to indicate we skip
        return arr, None, black_ratio

    tens = torch.from_numpy(arr.transpose(2,0,1))[None,...].to(device)
    with torch.no_grad():
        logits = model(tens)
        prob   = torch.sigmoid(logits)[0,0].cpu().numpy()
    mask = (prob>0.5).astype(np.uint8)
    return arr, mask, black_ratio

############################
# 6) LOAD LAND MASK
############################
def load_land_mask(fp, do_crop=True, size=(224,224)):
    if not os.path.exists(fp):
        return None
    pil = Image.open(fp).convert("L")
    if do_crop:
        pil = fixed_crop(pil)
    pil = pil.resize(size, Image.NEAREST)
    arr = np.array(pil, dtype=np.uint8) # {0=land,255=non-land}
    return arr

############################
# 7) PARTIAL COVERAGE + KNN
############################
def partial_coverage_classification(img_fp, cloud_model, knn_model, land_mask, device):
    arr_img, c_mask, black_ratio = predict_cloud_mask(cloud_model, img_fp, device, do_crop=True)
    if c_mask is None:
        # means we skip => too black
        return None, None, None, arr_img, None, black_ratio

    H,W,_ = arr_img.shape
    land_flat  = land_mask.reshape(-1)
    cloud_flat = c_mask.reshape(-1)
    valid_idx  = np.where((land_flat==255)&(cloud_flat==0))[0]
    coverage   = len(valid_idx)/(H*W)

    seg_map   = np.zeros((H,W,3), dtype=np.uint8)
    fractions = {"water":None, "thin sea ice":None, "sea ice":None}

    if coverage < CLOUD_THRESHOLD:
        return seg_map, fractions, coverage, arr_img, c_mask, black_ratio

    # do KNN
    flat_img = arr_img.reshape(-1,3)
    preds    = knn_model.predict(flat_img[valid_idx])
    labels   = np.full((H*W,), fill_value=-1, dtype=int)
    labels[valid_idx] = preds

    c_map = {
        -1: (0,0,0),
         0: (255,0,0),   # water => red
         1: (0,0,255),   # thin => blue
         2: (0,255,0)    # sea => green
    }
    for i in range(H*W):
        seg_map[i//W, i%W] = c_map.get(labels[i], (0,0,0))

    n_val    = len(valid_idx)
    frac_w   = np.sum(labels[valid_idx]==0)/n_val
    frac_t   = np.sum(labels[valid_idx]==1)/n_val
    frac_s   = np.sum(labels[valid_idx]==2)/n_val
    fractions= {
        "water": frac_w,
        "thin sea ice": frac_t,
        "sea ice": frac_s
    }
    return seg_map, fractions, coverage, arr_img, c_mask, black_ratio

############################
# 8) VISUALIZATION
############################
def visualize_result(fp, arr_img, c_mask, land_mask, seg_map, coverage, fractions, date, out_png, black_ratio):
    """
    2x2 => original, cloud, land, final classification
    If c_mask is None => means we skip => black
    """
    fig,axes= plt.subplots(2,2, figsize=(10,10))

    # original
    axes[0,0].imshow(arr_img)
    axes[0,0].set_title(os.path.basename(fp))
    axes[0,0].axis("off")

    # black ratio or cloud
    if c_mask is None:
        # means we skip => black
        axes[0,1].text(0.5,0.5, f"SKIPPED\nBlack ratio={black_ratio:.2f}",
                       ha="center", va="center", fontsize=14)
        axes[0,1].axis("off")

        # rest are blank
        for row in [1]:
            for col in [0,1]:
                axes[row,col].axis("off")

        fig.suptitle(f"Date => {date if date else '??'}, black ratio={black_ratio:.2f}", fontsize=14)
        plt.tight_layout()
        fig.savefig(out_png)
        plt.close(fig)
        return

    # c_mask
    cfrac= 100.0*c_mask.sum()/(arr_img.shape[0]*arr_img.shape[1])
    axes[0,1].imshow(c_mask, cmap="gray")
    axes[0,1].set_title(f"Cloud => ~{cfrac:.1f}%")
    axes[0,1].axis("off")

    # land
    land_show= land_mask.astype(np.float32)/255.0
    land_frac= 100.0*(land_mask==0).sum()/(land_mask.size)
    axes[1,0].imshow(land_show, cmap="gray")
    axes[1,0].set_title(f"Land => ~{land_frac:.1f}%")
    axes[1,0].axis("off")

    # final classification
    cov_pct= coverage*100.0
    axes[1,1].imshow(seg_map)
    if fractions["water"] is not None:
        w= fractions["water"]*100.0
        t= fractions["thin sea ice"]*100.0
        s= fractions["sea ice"]*100.0
        axes[1,1].set_title(f"Valid={cov_pct:.1f}%\nwater={w:.1f}%, thin={t:.1f}%, sea={s:.1f}%")
    else:
        axes[1,1].set_title(f"Valid={cov_pct:.1f}% => skip")
    axes[1,1].axis("off")

    fig.suptitle(f"Date => {date if date else '??'}, black ratio={black_ratio:.2f}", fontsize=14)
    plt.tight_layout()
    fig.savefig(out_png)
    plt.close(fig)

############################
# 9) MAIN
############################
def main():
    global device
    # A) Load cloud model
    print(f"[INFO] Loading model => {CLOUD_MODEL_PATH}")
    cloud_model = load_cloud_model(CLOUD_MODEL_PATH, device)

    # B) Load KNN
    if not os.path.exists(TRAINED_KNN_MODEL):
        raise FileNotFoundError(f"KNN model => {TRAINED_KNN_MODEL} not found")
    knn_model = joblib.load(TRAINED_KNN_MODEL)
    print(f"[INFO] KNN loaded => {TRAINED_KNN_MODEL}")

    # C) Land mask
    land_mask = load_land_mask(LAND_MASK_PATH, do_crop=True, size=IMG_SIZE)
    if land_mask is None:
        print(f"[ERROR] no land mask => {LAND_MASK_PATH}. Pipeline stops.")
        return

    # D) gather images
    year_folders= [os.path.join(SATELLITE_ROOT, d)
                   for d in os.listdir(SATELLITE_ROOT)
                   if os.path.isdir(os.path.join(SATELLITE_ROOT, d))]
    year_folders.sort()

    all_files=[]
    for fd in year_folders:
        fs= glob.glob(os.path.join(fd, "*.*"))
        fs= [f for f in fs if f.lower().endswith((".jpg",".png")) and "_classified" not in f.lower()]
        fs.sort()
        all_files.extend(fs)

    n_total= len(all_files)
    print(f"[INFO] Found {n_total} images in {SATELLITE_ROOT}...")

    # random subset for pdf
    random_subset= set(random.sample(all_files, min(MAX_VISUALS, n_total)))
    all_rows=[]
    pdf= PdfPages(OUTPUT_PDF)

    def extract_date(fp):
        base= os.path.basename(fp)
        ds= base[:8]
        try:
            d= datetime.strptime(ds, "%Y%m%d")
            return d
        except:
            return None

    # E) main loop
    for i,fp in enumerate(all_files, start=1):
        print(f"[{i}/{n_total}] => {os.path.basename(fp)}")
        dt= extract_date(fp)
        seg_map, fracs, coverage, arr_img, c_mask, black_ratio = partial_coverage_classification(
            fp, cloud_model, knn_model, land_mask, device
        )

        # if seg_map==None => means we skip => black
        skip_black = (seg_map is None)
        if skip_black:
            # store row => coverage=None, etc.
            row= {
                "filepath": fp,
                "date": dt,
                "coverage_fraction": None,
                "frac_water": None,
                "frac_thin_sea_ice": None,
                "frac_sea_ice": None,
                "classified_output": None,
                "black_ratio": black_ratio
            }
            all_rows.append(row)
            # maybe also produce a figure
            if fp in random_subset:
                out_png= os.path.join(VISUALS_FOLDER, os.path.basename(fp)+"_inspect.png")
                visualize_result(fp, arr_img, None, land_mask, None, None, {}, dt, out_png, black_ratio)
                fig= plt.figure(figsize=(10,10))
                ax= fig.add_subplot(111)
                insp_img= Image.open(out_png)
                ax.imshow(insp_img)
                ax.set_title(f"{os.path.basename(fp)} => SKIPPED black={black_ratio:.2f}")
                ax.axis("off")
                pdf.savefig(fig)
                plt.close(fig)
            continue

        out_seg=None
        if fracs["water"] is not None:
            out_seg = fp+"_classified.png"
            Image.fromarray(seg_map).save(out_seg)

        row= {
            "filepath": fp,
            "date": dt,
            "coverage_fraction": coverage,
            "frac_water": fracs["water"],
            "frac_thin_sea_ice": fracs["thin sea ice"],
            "frac_sea_ice": fracs["sea ice"],
            "classified_output": out_seg,
            "black_ratio": black_ratio
        }
        all_rows.append(row)

        if fp in random_subset:
            out_png= os.path.join(VISUALS_FOLDER, os.path.basename(fp)+"_inspect.png")
            visualize_result(fp, arr_img, c_mask, land_mask, seg_map, coverage, fracs, dt, out_png, black_ratio)

            fig= plt.figure(figsize=(10,10))
            ax= fig.add_subplot(111)
            insp_img= Image.open(out_png)
            ax.imshow(insp_img)
            ax.set_title(f"{os.path.basename(fp)} => coverage={coverage*100:.1f}% black={black_ratio:.2f}")
            ax.axis("off")
            pdf.savefig(fig)
            plt.close(fig)

    pdf.close()
    # output CSV
    df= pd.DataFrame(all_rows)
    df.sort_values("date", na_position="last", inplace=True)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved partial coverage CSV => {OUTPUT_CSV}")
    print(f"Created PDF => {OUTPUT_PDF}")
    print("\n=== Pipeline Complete ===")

if __name__=="__main__":
    main()


In [ ]:
# 9) Seasonal cycle of “valid coverage” with month labels
df["doy"] = df["date"].dt.dayofyear
# compute multi‑year mean of coverage_fraction, smoothed
clim = (
    df.groupby("doy")["coverage_fraction"]
      .mean()
      .rolling(window=7, center=True, min_periods=1)
      .mean()
)

# build month ticks on the x‑axis
months = pd.date_range("2020-01-01","2020-12-31",freq="MS")
pos    = months.dayofyear
labels = months.strftime("%b")

plt.figure(figsize=(10,4))
plt.plot(clim.index, clim.values, marker="o", ms=4, color="tab:blue")
plt.title("Seasonal Cycle: Mean Valid Coverage Fraction\n(non‑cloud, non‑black pixels)")
plt.xlabel("Month")
plt.ylabel("Mean Valid Coverage Fraction")
plt.xticks(pos, labels, rotation=0)
plt.xlim(1, 366)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


# Check how many bands my photos have

In [ ]:

import rasterio

with rasterio.open("../data/satellite/aqua/2016/20160612_AQUA.jpg") as src:
    print("Band count:", src.count)
    print("Width x Height:", src.width, "x", src.height)
    print("CRS:", src.crs)



In [ ]:
import requests

# --- 1. Obtain an OAuth Bearer Token ---
# Replace these with your CDSE OAuth client credentials.
client_id = "sh-d455e527-0659-4a60-95e7-8c66b8c56dce"
client_secret = "08idQHEEARpwS5FDZ1kYdIOL2I7CqmQm"
# Use the proper OAuth token endpoint (do not change the processing endpoint)
token_url = "https://auth.copernicus.eu/oauth/token"

payload = {"grant_type": "client_credentials"}
# Send the payload as JSON, with the proper header
token_response = requests.post(
    token_url,
    json=payload,
    auth=(client_id, client_secret),
    headers={"Content-Type": "application/json"}
)
if token_response.status_code != 200:
    raise Exception(f"Failed to obtain OAuth token: {token_response.status_code} {token_response.text}")

access_token = token_response.json()['access_token']
# Set up a session with the Authorization header using the retrieved token.
oauth = requests.Session()
oauth.headers.update({
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
})
print("Obtained OAuth token.")

# --- 2. Define AOI as a WKT string ---
# Using Uummannaq Bay coordinates (order: [minX, minY, maxX, maxY])
footprint_wkt = (
    "POLYGON ((-52.34915408749325 70.64081665526308, "
    "-51.93360986810109 70.64081665526308, "
    "-51.93360986810109 70.765296122774, "
    "-52.34915408749325 70.765296122774, "
    "-52.34915408749325 70.64081665526308))"
)
print("AOI in WKT:", footprint_wkt)

# --- 3. Define the evalscript (True Color example) ---
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B02", "B03", "B04"],
    output: {
      bands: 3,
      sampleType: "AUTO", // scales the output values from [0,1] to [0,255].
    },
  }
}

function evaluatePixel(sample) {
  return [2.5 * sample.B04, 2.5 * sample.B03, 2.5 * sample.B02]
}
"""

# --- 4. Build the processing request payload ---
# Here we update the time range to 2019-01-01 to 2025-04-16 and use the Uummannaq Bay bounding box.
request_payload = {
    "input": {
        "bounds": {
            "properties": {"crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"},
            "bbox": [
                -52.34915408749325,  # minX
                70.64081665526308,   # minY
                -51.93360986810109,  # maxX
                70.765296122774      # maxY
            ],
        },
        "data": [
            {
                "type": "sentinel-2-l1c",
                "dataFilter": {
                    "timeRange": {
                        "from": "2025-04-13T00:00:00Z",
                        "to": "2025-04-16T23:59:59Z",
                    }
                },
            }
        ],
    },
    "output": {
        "width": 512,
        "height": 512,
    },
    "evalscript": evalscript,
}

# --- 5. Define the processing endpoint ---
# (Leave this endpoint as it is per your request)
process_url = "https://sh.dataspace.copernicus.eu/api/v1/process"

# --- 6. Send the processing request ---
response = oauth.post(process_url, json=request_payload)
if response.ok:
    print("Processing request succeeded. Saving output to file...")
    with open("true_color_output.tiff", "wb") as f:
        f.write(response.content)
    print("Output saved as true_color_output.tiff")
else:
    print("Processing request failed:", response.status_code, response.text)

In [ ]:
#!/usr/bin/env python3
"""
Download and export a true‑color composite for each Sentinel‑2 L1C tile over Uummannaq Bay
(using a public STAC endpoint, odc‑stac, and odc‑geo for COG export).
"""

# 1. Monkey‑patch for compatibility
if not hasattr(dask.typing, "Key"):
    dask.typing.Key = object
if not hasattr(np, "round_"):
    np.round_ = np.round


import os
import numpy as np
import xarray as xr
import dask.typing
from pystac_client import Client
from odc.stac import load
import odc.geo.xr

# 0. Public S3 access (no AWS credentials needed)
os.environ["AWS_NO_SIGN_REQUEST"] = "YES"

# 2. Connect to the public STAC endpoint
client = Client.open("https://earth-search.aws.element84.com/v1")
print("Connected to STAC endpoint.")

# 3. Define AOI & time window
geometry = {"type":"Polygon","coordinates":[[[-51.949263,70.628336],[-52.336121,70.628336],[-52.336121,70.788206],[-51.949263,70.788206],[-51.949263,70.628336]]]}
datetime_range = "2025-04-10T00:00:00Z/2025-04-10T23:59:59Z"
collection = "sentinel-2-l1c"

# 4. Search for tiles
search = client.search(
    collections=[collection],
    intersects=geometry,
    datetime=datetime_range
)
items = list(search.items())
print(f"Found {len(items)} items:")
for itm in items:
    print(" ", itm.id)

if not items:
    raise SystemExit("No tiles found for that day/AOI.")

# 5. For each tile, load & export a true‑color COG
for item in items:
    print(f"\nProcessing {item.id} ...")

    # Load just that one item into a Dataset
    ds = load([item], geopolygon=geometry, groupby=None, chunks={})
    print("  Loaded dataset:", ds)

    # Grab the single timestamp
    t = ds.time.values[0]

    # Preview bands in L1C are named "visual.1", "visual.2", "visual.3"
    red   = ds["red"].sel(time=t)
    green = ds["green"].sel(time=t)
    blue  = ds["blue"].sel(time=t)

    # Stack into one DataArray with a "band" dimension
    rgb = xr.concat([red, green, blue], dim="band")
    rgb.attrs["nodata"] = 0

    # Build a safe filename: "<item_id>_<timestamp>.tiff"
    ts = np.datetime_as_string(t, unit="s").replace(":", "")
    fname = f"{item.id}_{ts}.tiff"

    # Write out the COG
    print(f"  Writing {fname} ...")
    odc.geo.xr.write_cog(rgb, fname=fname, overwrite=True)
    print(f"  Saved {fname}")

print("\nAll done!")


# Transform to 8-bit R,G,B for Photoshop

In [ ]:
import rasterio
from rasterio.enums import Resampling

src = rasterio.open("true_color_2025-04-15T152328.tiff")
profile = src.profile.copy()
profile.update({
    "dtype": "uint8",
    "count": 3,
    "compress": "lzw"
})

with rasterio.open("rgb_8bit.tif", "w", **profile) as dst:
    for b in (1, 2, 3):
        band = src.read(b, out_dtype="float32")
        # Simple linear scaling:
        band = (band / 10000.0) * 255.0
        band = band.clip(0, 255).astype("uint8")
        dst.write(band, b)


# First Approach with Cloudsen and AWS

In [ ]:
#!/usr/bin/env python3
"""
Fast CloudSEN inference on Sentinel‑2 L1C (13 bands):
  • 224×224 RGB thumbnail (visual bands)
  • 224×224 cloud mask PNG
"""

# ─── 0) ENV & COMPAT ───────────────────────────────────────────────────────────
import os
os.environ["AWS_NO_SIGN_REQUEST"] = "YES"    # allow public S3 access without creds

import numpy as np
if not hasattr(np, "round_"):
    np.round_ = np.round  # restore deprecated alias if missing

# monkey‑patch dask.typing.Key if needed
try:
    import dask.typing
    if not hasattr(dask.typing, "Key"):
        dask.typing.Key = object
except ImportError:
    pass

import torch
from pystac_client import Client
from odc.stac import load
from PIL import Image
import segmentation_models_pytorch as smp

# ─── 1) DEVICE & MODEL ─────────────────────────────────────────────────────────
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"[INFO] Using device: {device}")

print("[INFO] Building UNet (MobileNetV2 encoder, 13 in‑channels)…")
model = smp.Unet(
    encoder_name="mobilenet_v2",
    encoder_weights=None,
    in_channels=13,
    classes=4
).to(device)

print("[INFO] Loading checkpoint UNetMobV2_V2.pt…")
ckpt   = torch.load("UNetMobV2_V2.pt", map_location=device)
state  = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
clean  = {k.removeprefix("module.").removeprefix("model."):v for k,v in state.items()}
model.load_state_dict(clean, strict=False)
model.eval()
print("[OK] Model ready.\n")

# ─── 2) STAC SEARCH ─────────────────────────────────────────────────────────────
print("[INFO] Connecting to STAC endpoint…")
client = Client.open("https://earth-search.aws.element84.com/v1")

geometry = {
  "type":"Polygon","coordinates":[[[-52.336121,70.788206],
                                  [-51.945564,70.788206],
                                  [-51.945564,70.628226],
                                  [-52.336121,70.628226],
                                  [-52.336121,70.788206]]]
}
dt_range  = "2025-04-10T00:00:00Z/2025-04-10T23:59:59Z"
collection = "sentinel-2-l1c"

print(f"[INFO] Searching {collection} tiles in {dt_range}…")
items = list(client.search(
    collections=[collection],
    intersects=geometry,
    datetime=dt_range
).items())
print(f"[INFO] Found {len(items)} tiles")

# dedupe: one tile per day
unique, seen = [], set()
for it in items:
    day = it.datetime.date()
    if day not in seen:
        seen.add(day)
        unique.append(it)
items = unique
print(f"[INFO] Reduced to {len(items)} tile(s) (one per day)")
if not items:
    raise SystemExit("[ERROR] No tiles to process.")

# ─── 3) PROCESS EACH TILE ───────────────────────────────────────────────────────
for item in items:
    ts = item.datetime.strftime("%Y%m%dT%H%M%S")
    print(f"\n[INFO] === {item.id} @ {item.datetime.date()} ===")

    # A) load with odc-stac
    try:
        print("   → Loading data via odc-stac…")
        ds = load([item], geopolygon=geometry, groupby=None, chunks={})
    except Exception as e:
        print(f"   [ERROR] STAC load failed: {e}")
        continue

    # grab the single time value
    t0 = ds.time.values[0]

    # … after you have `ds` and `ts` defined …

    # … inside your loop, after `ds = load(...)` and `t0 = ds.time.values[0]` …

    # pull out the raw DN values (uint16)
    r_raw = ds["red"].isel(time=0).values.astype(np.float32)
    g_raw = ds["green"].isel(time=0).values.astype(np.float32)
    b_raw = ds["blue"].isel(time=0).values.astype(np.float32)

    # simple linear stretch 0–10000 → 0–255
    scale = 255.0 / 10000.0
    r = np.clip(r_raw * scale, 0, 255).astype(np.uint8)
    g = np.clip(g_raw * scale, 0, 255).astype(np.uint8)
    b = np.clip(b_raw * scale, 0, 255).astype(np.uint8)

    # make a PIL image and save
    from PIL import Image
    rgb = Image.merge("RGB",
        ( Image.fromarray(r, mode="L"),
        Image.fromarray(g, mode="L"),
        Image.fromarray(b, mode="L") )
    )
    rgb.save(f"{item.id}_{ts}_truecolor.jpg", quality=100)
    print(f"[OK] Saved true‑color JPEG: {item.id}_{ts}_truecolor.jpg")





    # C) CloudSEN inference on downsampled 13 bands
    try:
        print("   → Preparing 13‑band array & downsampling…")
        bands = [v for v in ds.data_vars if not v.startswith("visual.")]
        full  = np.stack([ ds[b].isel(time=0).values for b in bands ], axis=0
               ).astype(np.float32) / 10000.0
        C,H,W = full.shape
        print(f"      • raw shape = {C}×{H}×{W}")

        # block‑average to ~1/4 resolution
        s = 4
        H4 = (H//s)*s
        W4 = (W//s)*s
        small = full[:, :H4, :W4].reshape(C, H4//s, s, W4//s, s).mean(axis=(2,4))
        h4,w4 = small.shape[1:]
        print(f"      • downsampled = {C}×{h4}×{w4}")

        # pad to 32 multiple for UNet
        H2 = ((h4 +31)//32)*32
        W2 = ((w4 +31)//32)*32
        pad2 = np.zeros((C, H2, W2), dtype=small.dtype)
        pad2[:, :h4, :w4] = small
        print(f"      • padded to = {C}×{H2}×{W2}")

        print("   → Stacking & running inference…")
        x = torch.from_numpy(pad2[None]).to(device)  # shape (1,13,H2,W2)
        with torch.no_grad():
            logits = model(x)
            probs  = torch.softmax(logits, dim=1)[0,1,:h4,:w4].cpu().numpy()

        mask = (probs > 0.5).astype(np.uint8) * 255
        mask_img = Image.fromarray(mask, mode="L").resize((224,224), Image.NEAREST)

        png = f"{item.id}_{ts}_mask224.png"
        mask_img.save(png)
        print(f"   [OK] Saved mask → {png}")

    except Exception as e:
        print(f"   [ERROR] Inference failed: {e}")

print("\n[INFO] All done!")


In [ ]:
#!/usr/bin/env python3
"""
Fast CloudSEN12 UNetMobV2_V2 inference: 
 • 224×224 true‑color JPEG 
 • 512×512→224×224 cloud mask PNG via sliding‐window + AMP
"""

import os
os.environ["AWS_NO_SIGN_REQUEST"] = "YES"    # public S3 OK

import numpy as np
if not hasattr(np, "round_"):
    np.round_ = np.round

# monkey‑patch dask.typing.Key if needed
try:
    import dask.typing
    if not hasattr(dask.typing, "Key"):
        dask.typing.Key = object
except ImportError:
    pass

import torch
from torch.cuda.amp import autocast
from pystac_client import Client
from odc.stac import load
from PIL import Image
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

# ─── 1) DEVICE & MODEL ─────────────────────────────────────────
# ── DEVICE & MODEL ───────────────────────────────────────────
device = torch.device("mps") if torch.backends.mps.is_available() \
         else torch.device("cpu")
print(f"[INFO] Device: {device}")

model = smp.Unet(
    encoder_name="mobilenet_v2",
    encoder_weights=None,   # ← specify by keyword
    in_channels=13,
    classes=4
).to(device)

print("[INFO] Loading checkpoint UNetMobV2_V2.pt…")
ckpt  = torch.load("UNetMobV2_V2.pt", map_location=device)
state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
clean = {k.removeprefix("module.").removeprefix("model."):v for k,v in state.items()}
model.load_state_dict(clean, strict=False)
model.eval()
print("[OK] Model ready.\n")

# ─── 2) STAC SEARCH ─────────────────────────────────────────────
client   = Client.open("https://earth-search.aws.element84.com/v1")
geometry = {
    "type":"Polygon","coordinates":[[[-52.336121,70.788206],
                                     [-51.945564,70.788206],
                                     [-51.945564,70.628226],
                                     [-52.336121,70.628226],
                                     [-52.336121,70.788206]]]
}
dt_range, coll = "2025-03-10T00:00:00Z/2025-03-20T23:59:59Z", "sentinel-2-l1c"
items = list(client.search(
    collections=[coll],
    intersects=geometry,
    datetime=dt_range
).items())

# dedupe → one tile per day
unique, seen = [], set()
for it in items:
    day = it.datetime.date()
    if day not in seen:
        seen.add(day)
        unique.append(it)
items = unique

print(f"[INFO] {len(items)} tile(s) to process.\n")
if not items:
    raise SystemExit("[ERROR] no tiles found.")

# ─── 3) BAND ORDER ───────────────────────────────────────────────
band_order = [
    "coastal",  # B01
    "blue",     # B02
    "green",    # B03
    "red",      # B04
    "rededge1", # B05
    "rededge2", # B06
    "rededge3", # B07
    "nir",      # B08
    "nir08",    # B8A
    "nir09",    # B09
    "cirrus",   # B10
    "swir16",   # B11
    "swir22",   # B12
]

# ─── 4) PROCESS EACH TILE ────────────────────────────────────────
for item in items:
    ts = item.datetime.strftime("%Y%m%dT%H%M%S")
    print(f"[INFO] {item.id} @ {item.datetime.date()}")

    # A) load tile
    try:
        print("   → Loading data via odc-stac…")
        ds = load([item], geopolygon=geometry, groupby=None, chunks={})
    except Exception as e:
        print(f"   [ERROR] STAC load failed: {e}")
        continue

    # quick sanity check
    missing = [b for b in band_order if b not in ds.data_vars]
    if missing:
        print(f"   [ERROR] Missing bands: {missing}")
        continue

    # B) true‑color JPEG (linear stretch 0–10000 → 0–255)
    r_raw = ds["red"].isel(time=0).values.astype(np.float32)
    g_raw = ds["green"].isel(time=0).values.astype(np.float32)
    b_raw = ds["blue"].isel(time=0).values.astype(np.float32)
    scale = 255.0 / 10000.0
    r = np.clip(r_raw * scale, 0, 255).astype(np.uint8)
    g = np.clip(g_raw * scale, 0, 255).astype(np.uint8)
    b = np.clip(b_raw * scale, 0, 255).astype(np.uint8)

    rgb = Image.merge("RGB", (
        Image.fromarray(r, mode="L"),
        Image.fromarray(g, mode="L"),
        Image.fromarray(b, mode="L"),
    ))
    jpg = f"{item.id}_{ts}_truecolor.jpg"
    rgb.resize((512,512), Image.BILINEAR).save(jpg, quality=95)
    print(f"   [OK] Saved JPEG → {jpg}")

    # C) CloudSEN inference on 4× downsampled 512×512 patches
    print("   → Preparing 13‑band array & downsampling…")
    full = np.stack(
        [ ds[b].isel(time=0).values for b in band_order ],
        axis=0
    ).astype(np.float32) / 10000.0

    C,H,W   = full.shape
    s       = 4
    H4,W4   = (H//s)*s, (W//s)*s
    small   = full[:, :H4, :W4].reshape(C, H4//s, s, W4//s, s).mean((2,4))
    h4,w4   = small.shape[1:]
    print(f"      • raw={C}×{H}×{W} → down={C}×{h4}×{w4}")

    # sliding‐window 512×512 patches
    patches = []
    for y0 in range(0, h4, 512):
        for x0 in range(0, w4, 512):
            p = small[:, y0:y0+512, x0:x0+512]
            ph,pw = p.shape[1:]
            if (ph,pw)!=(512,512):
                pad = np.zeros((C,512,512), dtype=p.dtype)
                pad[:, :ph, :pw] = p
                p = pad
            patches.append(p)

    batch = torch.from_numpy(np.stack(patches, 0)).to(device)

   # inference with mixed precision
    print("   → Running inference on patches…")
    with autocast(device_type=device.type), torch.no_grad():   # <<< added device_type
        logits = model(batch)
        probs  = torch.softmax(logits, dim=1)[:,1]  # N×512×512


    # reassemble full‐size mask and downsample to 224×224
    mask_full = np.zeros((h4, w4), dtype=np.uint8)
    idx = 0
    for y0 in range(0, h4, 512):
        for x0 in range(0, w4, 512):
            ph = min(512, h4 - y0); pw = min(512, w4 - x0)
            prob = probs[idx, :ph, :pw].cpu().numpy()
            mask_full[y0:y0+ph, x0:x0+pw] = (prob > 0.5).astype(np.uint8) * 255
            idx += 1

    mask224 = Image.fromarray(mask_full, mode="L")\
                   .resize((224,224), Image.NEAREST)
    png = f"{item.id}_{ts}_mask224.png"
    mask224.save(png)
    print(f"   [OK] Saved mask → {png}")

    # D) side‑by‑side plot
    fig, ax = plt.subplots(1,2,figsize=(6,3), sharey=True)
    ax[0].imshow(np.array(rgb.resize((224,224))), origin="upper")
    ax[0].set_title("RGB")
    ax[0].axis("off")
    ax[1].imshow(mask224, cmap="gray", origin="upper")
    ax[1].set_title("CloudMask")
    ax[1].axis("off")
    plt.tight_layout()
    plt.show()

print("\n[INFO] All done!")


In [ ]:
"""
Fast CloudSEN12 UNetMobV2_V2 inference: 
 • 224×224 true‑color JPEG 
 • 512×512→224×224 cloud mask PNG via sliding‐window + AMP
"""

import os
os.environ["AWS_NO_SIGN_REQUEST"] = "YES"  # public S3 OK

import numpy as np
if not hasattr(np, "round_"):
    np.round_ = np.round

# monkey‑patch dask.typing.Key if needed
try:
    import dask.typing
    if not hasattr(dask.typing, "Key"):
        dask.typing.Key = object
except ImportError:
    pass

import torch
from torch.cuda.amp import autocast
from pystac_client import Client
from odc.stac import load
from PIL import Image
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

# ─── 1) DEVICE & MODEL ─────────────────────────────────────────
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"[INFO] Device: {device}")

print("[INFO] Building UNet (MobileNetV2 encoder, 13 in‑channels)…")
model = smp.Unet(
    encoder_name="mobilenet_v2",
    encoder_weights=None,
    in_channels=13,
    classes=4
).to(device)

print("[INFO] Loading checkpoint UNetMobV2_V2.pt…")
ckpt  = torch.load("UNetMobV2_V2.pt", map_location=device)
state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
clean = {k.removeprefix("module.").removeprefix("model."):v for k,v in state.items()}
model.load_state_dict(clean, strict=False)
model.eval()
print("[OK] Model ready.\n")

# ─── 2) STAC SEARCH ─────────────────────────────────────────────
client   = Client.open("https://earth-search.aws.element84.com/v1")
geometry = {
    "type": "Polygon",
    "coordinates": [[
        [-52.336121,70.788206],
        [-51.945564,70.788206],
        [-51.945564,70.628226],
        [-52.336121,70.628226],
        [-52.336121,70.788206]
    ]]
}
dt_range, coll = "2025-03-10T00:00:00Z/2025-03-20T23:59:59Z", "sentinel-2-l1c"
items = list(client.search(
    collections=[coll],
    intersects=geometry,
    datetime=dt_range
).items())

# dedupe → one tile per day
unique, seen = [], set()
for it in items:
    day = it.datetime.date()
    if day not in seen:
        seen.add(day)
        unique.append(it)
items = unique

print(f"[INFO] {len(items)} tile(s) to process.\n")
if not items:
    raise SystemExit("[ERROR] no tiles found.")

# ─── 3) BAND ORDER ───────────────────────────────────────────────
band_order = [
    "coastal",  # B01
    "blue",     # B02
    "green",    # B03
    "red",      # B04
    "rededge1", # B05
    "rededge2", # B06
    "rededge3", # B07
    "nir",      # B08
    "nir08",    # B8A
    "nir09",    # B09
    "cirrus",   # B10
    "swir16",   # B11
    "swir22",   # B12
]

# ─── 4) PROCESS EACH TILE ────────────────────────────────────────
for item in items:
    ts = item.datetime.strftime("%Y%m%dT%H%M%S")
    print(f"[INFO] {item.id} @ {item.datetime.date()}")

    # A) load tile
    try:
        print("   → Loading data via odc-stac…")
        ds = load([item], geopolygon=geometry, groupby=None, chunks={})
    except Exception as e:
        print(f"   [ERROR] STAC load failed: {e}")
        continue

    # Quick sanity check
    missing = [b for b in band_order if b not in ds.data_vars]
    if missing:
        print(f"   [ERROR] Missing bands: {missing}")
        continue

    # B) true‑color JPEG (linear stretch 0–10000 → 0–255)
    r = np.clip(ds["red"].isel(time=0).values.astype(np.float32) * 255/10000, 0,255).astype(np.uint8)
    g = np.clip(ds["green"].isel(time=0).values.astype(np.float32) * 255/10000, 0,255).astype(np.uint8)
    b = np.clip(ds["blue"].isel(time=0).values.astype(np.float32) * 255/10000, 0,255).astype(np.uint8)

    rgb = Image.merge("RGB", (
        Image.fromarray(r, mode="L"),
        Image.fromarray(g, mode="L"),
        Image.fromarray(b, mode="L"),
    ))
    jpg = f"{item.id}_{ts}_truecolor.jpg"
    rgb.resize((512,512), Image.BILINEAR).save(jpg, quality=95)
    print(f"   [OK] Saved JPEG → {jpg}")

    # C) CloudSEN inference on downsampled 512×512 patches
    print("   → Preparing 13‑band array & downsampling…")
    full = np.stack([ds[b].isel(time=0).values for b in band_order], axis=0) \
             .astype(np.float32) / 10000.0
    C,H,W = full.shape
    print(f"      • raw shape = {C}×{H}×{W}")

    # block‑average to 1/4 resolution
    s = 4
    H4, W4 = (H//s)*s, (W//s)*s
    small = full[:, :H4, :W4].reshape(C, H4//s, s, W4//s, s).mean((2,4))
    h4, w4 = small.shape[1:]
    print(f"      • downsampled = {C}×{h4}×{w4}")

    # sliding‐window into 512×512 tiles
    patches = []
    for y0 in range(0, h4, 512):
        for x0 in range(0, w4, 512):
            p = small[:, y0:y0+512, x0:x0+512]
            ph, pw = p.shape[1:]
            if (ph, pw) != (512, 512):
                pad = np.zeros((C,512,512), dtype=p.dtype)
                pad[:, :ph, :pw] = p
                p = pad
            patches.append(p)

    batch = torch.from_numpy(np.stack(patches,0)).to(device)

    # inference with mixed precision
    print("   → Running inference on patches…")
    with autocast(), torch.no_grad():
        logits = model(batch)
        probs  = torch.softmax(logits, dim=1)[:,1]  # cloud‐prob

    # reassemble & downsample mask to 224×224
    mask_full = np.zeros((h4, w4), dtype=np.uint8)
    idx = 0
    for y0 in range(0, h4, 512):
        for x0 in range(0, w4, 512):
            ph = min(512, h4 - y0)
            pw = min(512, w4 - x0)
            mask_full[y0:y0+ph, x0:x0+pw] = (probs[idx,:ph,:pw].cpu().numpy()>0.5)*255
            idx += 1

    mask224 = Image.fromarray(mask_full, mode="L")\
                   .resize((224,224), Image.NEAREST)
    png = f"{item.id}_{ts}_mask224.png"
    mask224.save(png)
    print(f"   [OK] Saved mask → {png}")

    # D) side‑by‑side plot
    fig, ax = plt.subplots(1,2,figsize=(6,3), sharey=True)
    ax[0].imshow(np.array(rgb.resize((224,224))), origin="upper")
    ax[0].set_title("RGB")
    ax[0].axis("off")
    ax[1].imshow(mask224, cmap="gray", origin="upper")
    ax[1].set_title("CloudMask")
    ax[1].axis("off")
    plt.tight_layout()
    plt.show()

print("\n[INFO] All done!")  


In [ ]:
#!/usr/bin/env python3
"""
Fast CloudSEN12 UNetMobV2_V2 inference:
  • 224×224 true‑color JPEG
  • 512×512→224×224 cloud mask PNG via sliding‑window + AMP
  • automatic cleanup of all intermediates per tile
"""

import os
os.environ["AWS_NO_SIGN_REQUEST"] = "YES"  # public S3 OK

import gc
import numpy as np
if not hasattr(np, "round_"):
    np.round_ = np.round

# monkey‑patch dask.typing.Key if needed
try:
    import dask.typing
    if not hasattr(dask.typing, "Key"):
        dask.typing.Key = object
except ImportError:
    pass

import torch
from torch.cuda.amp import autocast
import torch.nn.functional as F
from pystac_client import Client
from odc.stac import load
from PIL import Image
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

# ─── 1) DEVICE & MODEL ─────────────────────────────────────────
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"[INFO] Device: {device}")

print("[INFO] Building UNet (MobileNetV2 encoder, 13 in‑channels)…")
model = smp.Unet(
    encoder_name="mobilenet_v2",
    encoder_weights=None,
    in_channels=13,
    classes=4
).to(device)

print("[INFO] Loading checkpoint UNetMobV2_V2.pt…")
ckpt  = torch.load("UNetMobV2_V2.pt", map_location=device)
state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
clean = {k.removeprefix("module.").removeprefix("model."): v for k, v in state.items()}
model.load_state_dict(clean, strict=False)
model.eval()
print("[OK] Model ready.\n")

# ─── 2) STAC SEARCH ─────────────────────────────────────────────
client   = Client.open("https://earth-search.aws.element84.com/v1")
geometry = {
    "type": "Polygon",
    "coordinates": [[
        [-52.336121,70.788206],
        [-51.945564,70.788206],
        [-51.945564,70.628226],
        [-52.336121,70.628226],
        [-52.336121,70.788206]
    ]]
}
dt_range, coll = "2025-04-10T00:00:00Z/2025-04-16T23:59:59Z", "sentinel-2-l1c"
items = list(client.search(
    collections=[coll],
    intersects=geometry,
    datetime=dt_range
).items())

# dedupe → one tile per day
unique, seen = [], set()
for it in items:
    day = it.datetime.date()
    if day not in seen:
        seen.add(day)
        unique.append(it)
items = unique

print(f"[INFO] {len(items)} tile(s) to process.\n")
if not items:
    raise SystemExit("[ERROR] no tiles found.")

# ─── 3) BAND ORDER ───────────────────────────────────────────────
band_order = [
    "coastal",   # B01
    "blue",      # B02
    "green",     # B03
    "red",       # B04
    "rededge1",  # B05
    "rededge2",  # B06
    "rededge3",  # B07
    "nir",       # B08
    "nir08",     # B8A
    "nir09",     # B09
    "cirrus",    # B10
    "swir16",    # B11
    "swir22",    # B12
]

# ─── 4) PROCESS EACH TILE ────────────────────────────────────────
for item in items:
    ts = item.datetime.strftime("%Y%m%dT%H%M%S")
    print(f"[INFO] {item.id} @ {item.datetime.date()}")

    # A) load tile
    try:
        print("   → Loading data via odc-stac…")
        ds = load([item], geopolygon=geometry, groupby=None, chunks={})
    except Exception as e:
        print(f"   [ERROR] STAC load failed: {e}")
        continue

    # sanity check
    missing = [b for b in band_order if b not in ds.data_vars]
    if missing:
        print(f"   [ERROR] Missing bands: {missing}")
        continue

    # B) true‑color JPEG (linear stretch 0–10000 → 0–255)
    r = np.clip(ds["red"].isel(time=0).values.astype(np.float32) * 255/10000, 0, 255).astype(np.uint8)
    g = np.clip(ds["green"].isel(time=0).values.astype(np.float32) * 255/10000, 0, 255).astype(np.uint8)
    b = np.clip(ds["blue"].isel(time=0).values.astype(np.float32) * 255/10000, 0, 255).astype(np.uint8)

    rgb = Image.merge("RGB", (
        Image.fromarray(r, mode="L"),
        Image.fromarray(g, mode="L"),
        Image.fromarray(b, mode="L"),
    ))
    jpg = f"{item.id}_{ts}_truecolor.jpg"
    rgb.resize((512,512), Image.BILINEAR).save(jpg, quality=95)
    print(f"   [OK] Saved JPEG → {jpg}")

    # C) CloudSEN inference on downsampled 512×512 patches
    print("   → Preparing 13‑band array & downsampling…")
    full = np.stack([ds[b].isel(time=0).values for b in band_order], axis=0) \
             .astype(np.float32) / 10000.0
    C,H,W = full.shape
    print(f"      • raw shape = {C}×{H}×{W}")

    # avg‑pool by 4
    s = 4
    H4, W4 = (H//s)*s, (W//s)*s
    full_t = torch.from_numpy(full[None]).to(device)  # 1×13×H×W
    small = F.avg_pool2d(full_t[..., :H4, :W4], kernel_size=4, stride=4)
    _,C,h4,w4 = small.shape
    print(f"      • downsampled = {C}×{h4}×{w4}")

    # sliding‑window patches
    patches = []
    for y0 in range(0, h4, 512):
        for x0 in range(0, w4, 512):
            p = small[..., y0:y0+512, x0:x0+512]
            ph,pw = p.shape[-2:]
            if (ph,pw) != (512,512):
                pad = torch.zeros((1,C,512,512), device=device)
                pad[..., :ph, :pw] = p
                p = pad
            patches.append(p)
    batch = torch.cat(patches, 0)  # N×13×512×512

    # inference with AMP
    print("   → Running inference on patches…")
    with autocast(), torch.no_grad():
        logits = model(batch)
        probs  = torch.softmax(logits, dim=1)[:,1]  # N×512×512

    # reassemble + resize mask
    print("   → Reassembling mask & resizing to 224×224…")
    mask_full = torch.zeros((h4, w4), dtype=torch.uint8, device=device)
    idx = 0
    for y0 in range(0, h4, 512):
        for x0 in range(0, w4, 512):
            ph = min(512, h4 - y0)
            pw = min(512, w4 - x0)
            mask_full[y0:y0+ph, x0:x0+pw] = (probs[idx, :ph, :pw] > 0.5).byte()*255
            idx += 1

    mask224 = Image.fromarray(mask_full.cpu().numpy(), mode="L") \
                   .resize((224,224), Image.NEAREST)
    png = f"{item.id}_{ts}_mask224.png"
    mask224.save(png)
    print(f"   [OK] Saved mask → {png}")

    # D) side‑by‑side plot
    fig, axes = plt.subplots(1,2,figsize=(6,3), sharey=True)
    axes[0].imshow(np.array(rgb.resize((224,224))), origin="upper")
    axes[0].set_title("RGB");    axes[0].axis("off")
    axes[1].imshow(mask224, cmap="gray", origin="upper")
    axes[1].set_title("CloudMask"); axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    # ─── CLEANUP ────────────────────────────────────────────────────
    del ds, full, full_t, small, patches, batch, logits, probs, mask_full, mask224
    # free GPU memory if on cuda or mps
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    # force Python GC
    gc.collect()

print("\n[INFO] All done!")


In [ ]:
#!/usr/bin/env python3
"""
Fast CloudSEN12 inference on Sentinel‑2 L1C (13 bands):
  • 224×224 true‑color JPEG
  • 512×512→224×224 cloud mask PNG via sliding‑window + AMP
  • pluggable UNetMobV2_V2 or DeepLabV3+ (EfficientNet‑B0) backbones
"""

import os
os.environ["AWS_NO_SIGN_REQUEST"] = "YES"  # allow public S3 access without creds

import numpy as np
if not hasattr(np, "round_"):
    np.round_ = np.round  # restore deprecated alias if missing

# monkey‑patch dask.typing.Key if needed
try:
    import dask.typing
    if not hasattr(dask.typing, "Key"):
        dask.typing.Key = object
except ImportError:
    pass

import torch
from torch.amp import autocast                        # ← changed
import torch.nn.functional as F
from pystac_client import Client
from odc.stac import load
from PIL import Image
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

# ─── USER SETTINGS ────────────────────────────────────────────────
# Choose between "UNetMobV2_V2" (your existing CloudSEN12 model)
# or "DeepLabV3Plus_EffB0" (a more powerful transformer‑like head).
MODEL_TYPE = "UNetMobV2_V2"  # or "DeepLabV3Plus_EffB0"
CHECKPOINT_FILE = "UNetMobV2_V2.pt"  # if DeepLabV3+, point to your .pt

# ─── 1) DEVICE & MODEL ─────────────────────────────────────────
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"[INFO] Device: {device}")

if MODEL_TYPE == "UNetMobV2_V2":
    print("[INFO] Building UNet (MobileNetV2 encoder, 13 in‑channels)…")
    model = smp.Unet(
        encoder_name="mobilenet_v2",
        encoder_weights=None,
        in_channels=13,
        classes=4
    ).to(device)

elif MODEL_TYPE == "DeepLabV3Plus_EffB0":
    print("[INFO] Building DeepLabV3+ (EfficientNet‑B0 encoder, 13 in‑channels)…")
    model = smp.DeepLabV3Plus(
        encoder_name="efficientnet-b0",
        encoder_weights=None,
        in_channels=13,
        classes=4
    ).to(device)

else:
    raise ValueError(f"Unknown MODEL_TYPE '{MODEL_TYPE}'")

print(f"[INFO] Loading checkpoint {CHECKPOINT_FILE!r}…")
ckpt   = torch.load(CHECKPOINT_FILE, map_location=device)
state  = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
clean  = {k.removeprefix("module.").removeprefix("model."): v for k, v in state.items()}
model.load_state_dict(clean, strict=False)
model.eval()
print("[OK] Model ready.\n")

# ─── 2) STAC SEARCH ─────────────────────────────────────────────
client   = Client.open("https://earth-search.aws.element84.com/v1")
geometry = {
    "type": "Polygon",
    "coordinates": [[
        [-52.336121,70.788206],
        [-51.945564,70.788206],
        [-51.945564,70.628226],
        [-52.336121,70.628226],
        [-52.336121,70.788206]
    ]]
}
dt_range, coll = "2024-01-01T00:00:00Z/2024-12-31T23:59:59Z", "sentinel-2-l1c"
items = list(client.search(
    collections=[coll],
    intersects=geometry,
    datetime=dt_range
).items())

# dedupe → one tile per day
unique, seen = [], set()
for it in items:
    day = it.datetime.date()
    if day not in seen:
        seen.add(day)
        unique.append(it)
items = unique

print(f"[INFO] {len(items)} tile(s) to process.\n")
if not items:
    raise SystemExit("[ERROR] no tiles found.")

# ─── 3) BAND ORDER ───────────────────────────────────────────────
band_order = [
    "coastal",   # B01
    "blue",      # B02
    "green",     # B03
    "red",       # B04
    "rededge1",  # B05
    "rededge2",  # B06
    "rededge3",  # B07
    "nir",       # B08
    "nir08",     # B8A
    "nir09",     # B09
    "cirrus",    # B10
    "swir16",    # B11
    "swir22",    # B12
]

# ─── 4) PROCESS EACH TILE ────────────────────────────────────────
for item in items:
    ts = item.datetime.strftime("%Y%m%dT%H%M%S")
    print(f"[INFO] {item.id} @ {item.datetime.date()}")

    # A) load tile
    try:
        print("   → Loading data via odc-stac…")
        ds = load([item], geopolygon=geometry, groupby=None, chunks={})
    except Exception as e:
        print(f"   [ERROR] STAC load failed: {e}")
        continue

    # sanity check
    missing = [b for b in band_order if b not in ds.data_vars]
    if missing:
        print(f"   [ERROR] Missing bands: {missing}")
        continue

    # B) true‑color JPEG (linear stretch 0–10000 → 0–255)
    r = np.clip(ds["red"].isel(time=0).values.astype(np.float32) * 255/10000, 0,255).astype(np.uint8)
    g = np.clip(ds["green"].isel(time=0).values.astype(np.float32) * 255/10000, 0,255).astype(np.uint8)
    b = np.clip(ds["blue"].isel(time=0).values.astype(np.float32) * 255/10000, 0,255).astype(np.uint8)

    rgb = Image.merge("RGB", (
        Image.fromarray(r, mode="L"),
        Image.fromarray(g, mode="L"),
        Image.fromarray(b, mode="L"),
    ))
    jpg = f"{item.id}_{ts}_truecolor.jpg"
    rgb.resize((512,512), Image.BILINEAR).save(jpg, quality=95)
    print(f"   [OK] Saved JPEG → {jpg}")

    # C) Cloud mask inference on downsampled 512×512 patches
    print("   → Preparing 13‑band array & downsampling…")
    full = np.stack([ds[b].isel(time=0).values for b in band_order], axis=0) \
             .astype(np.float32) / 10000.0
    C,H,W = full.shape
    print(f"      • raw = {C}×{H}×{W}")

    # avg‑pool by 4
    s = 4
    H4, W4 = (H//s)*s, (W//s)*s
    full_t = torch.from_numpy(full[None]).to(device)  # 1×13×H×W
    small  = F.avg_pool2d(full_t[..., :H4, :W4], kernel_size=4, stride=4)
    _,C,h4,w4 = small.shape
    print(f"      • downsampled = {C}×{h4}×{w4}")

    # sliding‐window into 512×512 tiles
    patches = []
    for y0 in range(0, h4, 512):
        for x0 in range(0, w4, 512):
            p = small[..., y0:y0+512, x0:x0+512]
            ph, pw = p.shape[-2:]
            if (ph,pw)!=(512,512):
                pad=torch.zeros((C,512,512),device=device)
                pad[:,:ph,:pw]=blk; blk=pad
            patches.append(p)
    batch = torch.cat(patches, 0)  # N×13×512×512

    # inference with mixed precision
    # inference with mixed precision
    print("   → Running inference on patches…")
    with autocast(device_type=device.type), torch.no_grad():   # <<< added device_type
        logits = model(batch)
        probs  = torch.softmax(logits, dim=1)[:,1]  # N×512×512


    # reassemble + downsample mask to 224×224
    print("   → Reassembling mask & resizing to 224×224…")
    mask_full = torch.zeros((h4, w4), dtype=torch.uint8, device=device)
    idx = 0
    for y0 in range(0, h4, 512):
        for x0 in range(0, w4, 512):
            ph, pw = min(512, h4 - y0), min(512, w4 - x0)
            mask_full[y0:y0+ph, x0:x0+pw] = (probs[idx,:ph,:pw] > 0.5).byte()*255
            idx += 1

    mask224 = Image.fromarray(mask_full.cpu().numpy(), mode="L") \
                   .resize((224,224), Image.NEAREST)
    png = f"{item.id}_{ts}_mask224.png"
    mask224.save(png)
    print(f"   [OK] Saved mask → {png}")

    # D) side‑by‑side plot
    fig, axes = plt.subplots(1,2,figsize=(6,3), sharey=True)
    axes[0].imshow(np.array(rgb.resize((224,224))), origin="upper")
    axes[0].set_title("RGB");    axes[0].axis("off")
    axes[1].imshow(mask224, cmap="gray", origin="upper")
    axes[1].set_title("CloudMask"); axes[1].axis("off")
    plt.tight_layout()
    plt.show()

    # E) cleanup variables to free memory
    del ds, full, full_t, small, batch, logits, probs, mask_full
    if torch.backends.mps.is_available():             # ← changed
        torch.mps.empty_cache()
    else:
        torch.cuda.empty_cache()

print("\n[INFO] All done!")


# Make a landmask

In [ ]:
#!/usr/bin/env python3
"""
Make a 40 m RGB thumbnail (512×512) to paint your static land mask.
Chooses the clearest Sentinel‑2 tile (cloud_cover ≤ 20 %).
"""

import os, numpy as np, torch, torch.nn.functional as F
from pystac_client import Client
from odc.stac import load
from PIL import Image

os.environ["AWS_NO_SIGN_REQUEST"] = "YES"

SEARCH_AOI = {"type":"Polygon","coordinates":[[
    [-52.336121,70.788206],[-51.945564,70.788206],
    [-51.945564,70.628226],[-52.336121,70.628226],
    [-52.336121,70.788206]]]}
DATE_RANGE = "2024-01-01T00:00:00Z/2024-12-31T23:59:59Z"   # feel free to change
MAX_CC     = 20.0                                           # % cloud threshold

# 1) find the clearest tile ---------------------------------------------
client = Client.open("https://earth-search.aws.element84.com/v1")
items  = sorted(client.search(
    collections=["sentinel-2-l1c"],
    intersects=SEARCH_AOI,
    datetime=DATE_RANGE,
).items(), key=lambda it: it.properties.get("eo:cloud_cover", 100.0))

item = next((it for it in items
             if it.properties.get("eo:cloud_cover", 100.0) <= MAX_CC), None)
if item is None:
    raise SystemExit(f"No tile with cloud_cover ≤ {MAX_CC}% found in range.")

cc = item.properties.get("eo:cloud_cover", "N/A")
print(f"[INFO] Using {item.id}  (cloud_cover = {cc} %)")

# 2) load R‑G‑B, down‑sample to 40 m -------------------------------------
ds   = load([item], geopolygon=SEARCH_AOI)
R,G,B = [ds[c].isel(time=0).values.astype(np.float32) for c in ("red","green","blue")]
scale = 255.0/10000.0
rgb   = np.stack([np.clip(c*scale,0,255) for c in (R,G,B)], 0)          # 3×H×W

H,W   = R.shape
H4,W4 = (H//4)*4, (W//4)*4
rgb40 = F.avg_pool2d(torch.from_numpy(rgb[None, :, :H4, :W4]), 4, 4) \
            .squeeze(0).byte().numpy()                                  # 3×h×w uint8

# 3) save 512‑px template ------------------------------------------------
Image.fromarray(rgb40.transpose(1,2,0), "RGB") \
     .resize((512,512), Image.BILINEAR) \
     .save("landmask_template.png")

print("Template saved → landmask_template.png  (paint land=white, water=black)")


# Final Version - CloudSEN12, Water, Ice

In [ ]:
#!/usr/bin/env python3
"""
Fast CloudSEN12 UNetMobV2_V2 inference + sea‑ice / water masks
with user‑painted land mask and missing‑edge detection.

Outputs per tile:
  • *_mask512.png, *_ice512.png, *_water512.png, *_land512.png, *_overlay512.png
  • summary.csv with rich metrics incl. nodata.
"""

import os, gc, csv, matplotlib.pyplot as plt
os.environ["AWS_NO_SIGN_REQUEST"] = "YES"

import numpy as np
if not hasattr(np, "round_"): np.round_ = np.round
import torch, torch.nn.functional as F
from torch.amp import autocast
from pystac_client import Client
# monkey‑patch dask.typing.Key if needed
try:
    import dask.typing
    if not hasattr(dask.typing, "Key"):
        dask.typing.Key = object
except ImportError:
    pass

from odc.stac import load
from PIL import Image, ImageDraw
import segmentation_models_pytorch as smp

# ─── USER SETTINGS ─────────────────────────────────────────────
CHECKPOINT_FILE  = "UNetMobV2_V2.pt"
LANDMASK_FILE    = "landmask_template.png"      # white land, black water
NDSI_THR, NDWI_THR = 0.42, 0.05
NODATA_THR        = 0.05                        # 5 % ⇒ edge_gap flag
CSV_FILE          = "summary.csv"

# ─── DEVICE & MODEL ────────────────────────────────────────────
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"[INFO] Device: {device}")

model = smp.Unet("mobilenet_v2", encoder_weights=None, in_channels=13, classes=4).to(device)
ckpt  = torch.load(CHECKPOINT_FILE, map_location=device)
model.load_state_dict({k.removeprefix("module.").removeprefix("model."): v
                       for k,v in ckpt.get("model_state_dict", ckpt).items()}, strict=False)
model.eval(); print("[OK] Model ready.\n")

# ─── STAC SEARCH ───────────────────────────────────────────────
client = Client.open("https://earth-search.aws.element84.com/v1")
SEARCH_AOI = {"type":"Polygon","coordinates":[[
    [-52.336121,70.788206],[-51.945564,70.788206],
    [-51.945564,70.628226],[-52.336121,70.628226],
    [-52.336121,70.788206]]]}
DATE_RANGE = "2024-01-01T00:00:00Z/2024-12-31T23:59:59Z"

items = list(client.search(collections=["sentinel-2-l1c"],
                           intersects=SEARCH_AOI, datetime=DATE_RANGE).items())
items = {it.datetime.date(): it for it in items}.values()
print(f"[INFO] {len(items)} tile(s) to process.\n")
if not items: raise SystemExit("[ERROR] no tiles found.")

# ─── CONSTANTS & HELPERS ───────────────────────────────────────
bands = ["coastal","blue","green","red","rededge1","rededge2","rededge3",
         "nir","nir08","nir09","cirrus","swir16","swir22"]
g_idx, nir_idx, sw_idx = map(bands.index, ["green","nir","swir16"])
class_color = { "ice": (0,255,255), "water":(0,0,255),
                "cloud":(200,200,200), "land":(150,75,0), "nodata":(255,0,255)}

def preview(mask: np.ndarray) -> Image.Image:
    hi = Image.fromarray((mask*255).astype(np.uint8), "L")
    return hi.resize((2048,2048), Image.NEAREST).resize((512,512), Image.BILINEAR)

def colour_overlay(rgb: Image.Image,
                   ice, water, cloud, land, nodata) -> Image.Image:
    base = rgb.convert("RGBA")
    overlay_arr = np.zeros((*base.size[::-1], 4), dtype=np.uint8)
    for mask,label in [(ice,"ice"),(water,"water"),
                       (cloud,"cloud"),(land,"land"),(nodata,"nodata")]:
        mask_np = np.array(Image.fromarray((mask*255).astype(np.uint8))
                           .resize(base.size, Image.NEAREST)) > 127
        overlay_arr[mask_np] = (*class_color[label], 120)
    return Image.alpha_composite(base, Image.fromarray(overlay_arr,"RGBA")).convert("RGB")

# ─── CSV HEADER ────────────────────────────────────────────────
header = ["tile_id","timestamp",
          "ice_px","water_px","cloud_px","land_px","nodata_px","unknown_px",
          "ice_pct","water_pct","cloud_pct","land_pct","nodata_pct",
          "mean_ndsi_ice","mean_ndwi_water",
          "eo_cloud_cover","sun_elev","sun_azim","edge_gap"]
with open(CSV_FILE,"w",newline="") as fcsv:
    csv.writer(fcsv).writerow(header)

# ─── MAIN LOOP ────────────────────────────────────────────────
for it in items:
    ts   = it.datetime.strftime("%Y%m%dT%H%M%S")
    meta = it.properties
    print(f"\n[INFO] {it.id}  {ts}")

    ds = load([it], geopolygon=SEARCH_AOI, groupby=None, chunks={})

    # RGB 10 m → 512 px
    R,G,B = [ds[c].isel(time=0).values.astype(np.float32) for c in ("red","green","blue")]
    scale = 255.0 if R.max()<=1 else 255.0/10000.0
    rgb512 = Image.merge("RGB",[Image.fromarray(np.clip(c*scale,0,255).astype(np.uint8),"L")
                                 for c in (R,G,B)]).resize((512,512), Image.BILINEAR)

    # cube 40 m
    cube = np.stack([ds[b].isel(time=0).values for b in bands]).astype(np.float32)
    if cube.max()>1.1: cube/=10000.0
    C,H,W = cube.shape; H4,W4 = (H//4)*4,(W//4)*4
    small = F.avg_pool2d(torch.from_numpy(cube[None])[...,:H4,:W4],4,4).squeeze(0)
    _,h4,w4 = small.shape
    s_np = small.cpu().numpy()

    # nodata (all‑zero)
    nodata = (s_np.sum(0) < 1e-6)
    nodata_px = int(nodata.sum()); nodata_pct = nodata_px/(h4*w4)
    edge_gap  = int(nodata_pct >= NODATA_THR)
    if edge_gap:
        print(f"   [WARN] nodata {nodata_pct*100:.1f}% → edge gap flagged")

    # land mask
    land = (np.array(Image.open(LANDMASK_FILE).convert("L")
                     .resize((w4,h4), Image.NEAREST)) > 127)

    # cloud inference
    cloud = np.zeros((h4,w4),bool); pcs,coords=[],[]
    for y0 in range(0,h4,512):
        for x0 in range(0,w4,512):
            p=small[:,y0:y0+512,x0:x0+512]; ph,pw=p.shape[-2:]
            if (ph,pw)!=(512,512):
                pad=torch.zeros((C,512,512),device=device); pad[:,:ph,:pw]=p; p=pad
            pcs.append(p); coords.append((y0,x0,ph,pw))
    if pcs:
        with autocast(device_type=device.type), torch.no_grad():
            pr = torch.softmax(model(torch.stack(pcs).to(device)),1)[:,1].cpu().numpy()
        for (y0,x0,ph,pw),p in zip(coords,pr):
            cloud[y0:y0+ph,x0:x0+pw] = p[:ph,:pw] > 0.5

    # ND indices & classes
    ndsi = (s_np[g_idx]-s_np[sw_idx])/(s_np[g_idx]+s_np[sw_idx]+1e-6)
    ndwi = (s_np[g_idx]-s_np[nir_idx])/(s_np[g_idx]+s_np[nir_idx]+1e-6)
    ice   = (ndsi>NDSI_THR)&~cloud&~land&~nodata
    water = (ndwi>NDWI_THR)&~ice&~cloud&~land&~nodata

    # stats
    total = h4*w4
    counts = {"ice":int(ice.sum()),"water":int(water.sum()),
              "cloud":int(cloud.sum()),"land":int(land.sum()),"nodata":nodata_px}
    unknown = total - sum(counts.values())
    pct = {k:v/total for k,v in counts.items()}
    mean_ndsi = float(np.nanmean(ndsi[ice]))  if counts["ice"]   else np.nan
    mean_ndwi = float(np.nanmean(ndwi[water])) if counts["water"] else np.nan

    # previews & overlay
    prev = {n:preview(m) for n,m in [("cloud",cloud),("ice",ice),
                                     ("water",water),("land",land),("nodata",nodata)]}
    overlay = colour_overlay(rgb512, ice, water, cloud, land, nodata)
    overlay.save(f"{it.id}_{ts}_overlay512.png")

    # live 2×3 panel
    fig,ax = plt.subplots(2,3,figsize=(13,8))
    ax[0,0].imshow(rgb512);       ax[0,0].set_title("RGB"); ax[0,0].axis("off")
    ax[0,1].imshow(prev["cloud"],cmap="gray"); ax[0,1].set_title("Cloud"); ax[0,1].axis("off")
    ax[0,2].imshow(prev["land"],cmap="gray");  ax[0,2].set_title("Land");  ax[0,2].axis("off")
    ax[1,0].imshow(prev["ice"],cmap="gray");   ax[1,0].set_title("Sea‑Ice");ax[1,0].axis("off")
    ax[1,1].imshow(prev["water"],cmap="gray"); ax[1,1].set_title("Water");  ax[1,1].axis("off")a
    ax[1,2].imshow(overlay);      ax[1,2].set_title("Overlay"); ax[1,2].axis("off")
    plt.suptitle(f"{it.id}  {ts}", fontsize=11); plt.tight_layout(); plt.show()

    # save previews
    prev["cloud"].save(f"{it.id}_{ts}_mask512.png")
    prev["ice"].save  (f"{it.id}_{ts}_ice512.png")
    prev["water"].save(f"{it.id}_{ts}_water512.png")
    prev["land"].save (f"{it.id}_{ts}_land512.png")

    # CSV append
    with open(CSV_FILE,"a",newline="") as fcsv:
        csv.writer(fcsv).writerow([
            it.id, ts,
            counts["ice"], counts["water"], counts["cloud"], counts["land"],
            nodata_px, unknown,
            round(pct["ice"],4), round(pct["water"],4),
            round(pct["cloud"],4), round(pct["land"],4),
            round(nodata_pct,4),
            round(mean_ndsi,4) if not np.isnan(mean_ndsi) else "",
            round(mean_ndwi,4) if not np.isnan(mean_ndwi) else "",
            meta.get("eo:cloud_cover",""), meta.get("sat:solar_elevation",""),
            meta.get("sat:solar_azimuth",""), edge_gap
        ])

    # cleanup
    del ds,cube,small; gc.collect()
    torch.mps.empty_cache() if torch.backends.mps.is_available() else torch.cuda.empty_cache()

print(f"\n[INFO] All done! Summary written to {CSV_FILE}")


In [ ]:
# --- EDA with robust export -------------------------------------------
import pandas as pd, numpy as np, plotly.express as px, plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"

CSV_FILE   = "summary.csv"
CLOUD_MAX  = 0.03
NODATA_MAX = 0.02
ROLL_DAYS  = 7
PX_AREA_KM2 = (40*40)/1e6   # km2 per 40 m px

# helper ----------------------------------------------------------------
def safe_export(fig, stem):
    png = f"{stem}.png"
    html = f"{stem}.html"
    try:
        fig.write_image(png, scale=2, engine="kaleido")
        print(f"PNG  -> {png}")
    except Exception as e:
        print(f"[WARN] Kaleido failed: {e}.  Falling back to HTML.")
        fig.write_html(html, include_plotlyjs="cdn")
        print(f"HTML -> {html}")

# load + QC -------------------------------------------------------------
df = pd.read_csv(CSV_FILE, parse_dates=["timestamp"])
if "edge_gap" not in df.columns:
    df["edge_gap"] = 0
df = df[(df["edge_gap"] == 0) &
        (df["cloud_pct"] <= CLOUD_MAX) &
        (df["nodata_pct"] <= NODATA_MAX)].copy()

df["month"] = df["timestamp"].dt.month_name()
df["doy"]   = df["timestamp"].dt.dayofyear
df["week"]  = df["timestamp"].dt.isocalendar().week.astype(int)
df["ice_km2"]   = df["ice_px"]   * PX_AREA_KM2
df["water_km2"] = df["water_px"] * PX_AREA_KM2

print(f"Tiles kept: {len(df)}")

# 1. Daily sea‑ice % ----------------------------------------------------
fig1 = go.Figure()
fig1.add_scatter(x=df["timestamp"], y=df["ice_pct"]*100,
                 mode="lines+markers", marker_size=4,
                 line=dict(color="#66c2ff", width=1),
                 name="Daily %")
fig1.add_scatter(x=df["timestamp"],
                 y=df["ice_pct"].rolling(ROLL_DAYS, 1).mean()*100,
                 mode="lines", line=dict(color="#004c99", width=3),
                 name=f"{ROLL_DAYS}-day mean")
thr = df[df["ice_pct"] > 0.5]
if not thr.empty:
    for txt,row in [("freeze-up", thr.iloc[0]), ("break-up", thr.iloc[-1])]:
        fig1.add_annotation(x=row["timestamp"], y=row["ice_pct"]*100,
                            text=txt, showarrow=True, arrowhead=3)
fig1.update_layout(title="Daily Sea Ice Cover 2024",
                   xaxis_title="Date", yaxis_title="Sea Ice (%)",
                   template="plotly_white",
                   legend=dict(orientation="h", yanchor="bottom", y=1.02))
safe_export(fig1, "sea_ice_daily")

# 2. Monthly box --------------------------------------------------------
order = ["January","February","March","April","May","June",
         "July","August","September","October","November","December"]
fig2 = px.box(df, x="month", y=df["ice_pct"]*100, color="month",
              category_orders={"month": order},
              title="Sea Ice % by Month (2024)",
              labels={"month":"", "y":"Sea Ice (%)"},
              template="plotly_white")
fig2.update_traces(marker_size=4)
safe_export(fig2, "sea_ice_box")

# 3. Calendar heat map --------------------------------------------------
pivot = df.pivot_table(index="doy", columns="week",
                       values="ice_pct") * 100
fig3 = px.imshow(pivot, origin="lower", aspect="auto",
                 color_continuous_scale="Blues",
                 labels=dict(x="ISO Week", y="Day of Year",
                             color="Sea Ice (%)"),
                 title="Calendar Heat Map – Sea Ice % 2024",
                 template="plotly_white")
safe_export(fig3, "sea_ice_calendar")

# 4. Cumulative area ----------------------------------------------------
cum = (df.sort_values("timestamp")
         .assign(ice_cum=lambda d:d["ice_km2"].cumsum(),
                 water_cum=lambda d:d["water_km2"].cumsum()))
cum["ratio"] = cum["ice_cum"] / (cum["ice_cum"] + cum["water_cum"])
fig4 = go.Figure()
fig4.add_scatter(x=cum["timestamp"], y=cum["ice_cum"],
                 mode="lines", line_color="#00e5ff", name="Ice km²")
fig4.add_scatter(x=cum["timestamp"], y=cum["water_cum"],
                 mode="lines", line_color="#0077ff", name="Water km²")
fig4.add_scatter(x=cum["timestamp"], y=cum["ratio"],
                 mode="lines", line=dict(color="black", dash="dot"),
                 name="Ice fraction", yaxis="y2")
fig4.update_layout(title="Cumulative Ice and Water Area 2024",
                   template="plotly_white",
                   yaxis_title="Cumulative km²",
                   yaxis2=dict(title="Ice Fraction",
                               overlaying="y", side="right", range=[0,1]),
                   legend=dict(orientation="h", yanchor="bottom", y=1.02))
safe_export(fig4, "cum_area")

# show inline (optional)
for fig in (fig1, fig2, fig3, fig4):
    fig.show()
# ----------------------------------------------------------------------


# Stitching Example 

In [ ]:
#!/usr/bin/env python3
"""
Fast CloudSEN12 UNetMobV2_V2 inference + sea‑ice / water masks
with user‑painted land mask and missing‑edge detection.

Outputs per date:
  • stitched *_mask512.png, *_ice512.png, *_water512.png, *_land512.png, *_overlay512.png
  • summary.csv with rich metrics incl. nodata.
"""

import os, gc, csv, matplotlib.pyplot as plt
os.environ["AWS_NO_SIGN_REQUEST"] = "YES"

import numpy as np
if not hasattr(np, "round_"): np.round_ = np.round
import torch, torch.nn.functional as F
from torch.amp import autocast
from pystac_client import Client
# monkey‑patch dask.typing.Key if needed
try:
    import dask.typing
    if not hasattr(dask.typing, "Key"):
        dask.typing.Key = object
except ImportError:
    pass

from odc.stac import load
from PIL import Image
import segmentation_models_pytorch as smp
from shapely.geometry import box
from shapely.ops import unary_union

# ─── USER SETTINGS ─────────────────────────────────────────────
CHECKPOINT_FILE  = "UNetMobV2_V2.pt"
LANDMASK_FILE    = "landmask_template.png"
NDSI_THR, NDWI_THR = 0.42, 0.05
NODATA_THR        = 0.05
CSV_FILE          = "summary.csv"
DATE_RANGE        = "2024-02-15T00:00:00Z/2024-02-15T23:59:59Z"

# original AOI
SEARCH_AOI = {
    "type":"Polygon",
    "coordinates":[[ 
        [-52.336121,70.788206],[-51.945564,70.788206],
        [-51.945564,70.628226],[-52.336121,70.628226],
        [-52.336121,70.788206]
    ]]
}

# expand it by 0.2° west and 0.2° north
coords = SEARCH_AOI["coordinates"][0]
lons = [c[0] for c in coords]; lats = [c[1] for c in coords]
west_buffer, east_buffer = 0.00, 0.00
south_buffer, north_buffer = 0.00, 0.00
min_lon = min(lons) - west_buffer
max_lon = max(lons) + east_buffer
min_lat = min(lats) - south_buffer
max_lat = max(lats) + north_buffer

SEARCH_AOI_EXPANDED = {
    "type":"Polygon",
    "coordinates":[[ 
        [min_lon, min_lat],
        [max_lon, min_lat],
        [max_lon, max_lat],
        [min_lon, max_lat],
        [min_lon, min_lat]
    ]]
}

# ─── DEVICE & MODEL ────────────────────────────────────────────
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"[INFO] Device: {device}")
model = smp.Unet("mobilenet_v2", encoder_weights=None, in_channels=13, classes=4).to(device)
ckpt  = torch.load(CHECKPOINT_FILE, map_location=device)
model.load_state_dict({
    k.removeprefix("module.").removeprefix("model."): v
    for k,v in ckpt.get("model_state_dict", ckpt).items()
}, strict=False)
model.eval()
print("[OK] Model ready.\n")

# ─── STAC SEARCH OVER EXPANDED AOI ────────────────────────────
client = Client.open("https://earth-search.aws.element84.com/v1")
items_raw = list(client.search(
    collections=["sentinel-2-l1c"],
    intersects=SEARCH_AOI_EXPANDED,
    datetime=DATE_RANGE
).items())

# group by acquisition date
by_date = {}
for it in items_raw:
    d = it.datetime.date()
    by_date.setdefault(d, []).append(it)

print(f"[INFO] {len(by_date)} unique dates found.\n")
if not by_date:
    raise SystemExit("[ERROR] no tiles found.")

# ─── DEBUG: first date ► count, raw previews, full stitch ─────────
first_date, first_tiles = next(iter(sorted(by_date.items())))
print(f"[DEBUG] {first_date}: found {len(first_tiles)} tiles, previewing all and stitching")

# preview each raw tile (uncropped)
fig, axs = plt.subplots(1, len(first_tiles), figsize=(5*len(first_tiles),5))
for ax, it in zip(axs, first_tiles):
    try:
        ds0 = load([it], groupby=None, chunks={})
        R,G,B = [ds0[c].isel(time=0).values.astype(np.float32) for c in ("red","green","blue")]
        scale = 255.0 if R.max()<=1 else 255.0/10000.0
        rgb0 = np.stack([
            np.clip(R*scale,0,255).astype(np.uint8),
            np.clip(G*scale,0,255).astype(np.uint8),
            np.clip(B*scale,0,255).astype(np.uint8)
        ], axis=-1)
        ax.imshow(rgb0); ax.set_title(it.id, fontsize=8)
    except:
        ax.text(0.5,0.5,"ERROR",ha="center",va="center")
        ax.set_title(it.id+"\n(failed)", fontsize=8)
    ax.axis("off")
plt.suptitle(f"Raw tiles for {first_date}")
plt.tight_layout(); plt.show()

# stitch **all** first_tiles together and crop back
try:
    ds_st = load(first_tiles, geopolygon=SEARCH_AOI, groupby=None, chunks={})
    R,G,B = [ds_st[c].isel(time=0).values.astype(np.float32) for c in ("red","green","blue")]
    scale = 255.0 if R.max()<=1 else 255.0/10000.0
    rgb_st = np.stack([
        np.clip(R*scale,0,255).astype(np.uint8),
        np.clip(G*scale,0,255).astype(np.uint8),
        np.clip(B*scale,0,255).astype(np.uint8)
    ], axis=-1)
    plt.figure(figsize=(6,6))
    plt.imshow(rgb_st); plt.title(f"Stitched mosaic for {first_date}"); plt.axis("off"); plt.show()
except Exception as e:
    print(f"[WARN] Stitched load failed for {first_date}: {e}")

# ─── CONSTANTS & HELPERS ───────────────────────────────────────
bands = ["coastal","blue","green","red","rededge1","rededge2","rededge3",
         "nir","nir08","nir09","cirrus","swir16","swir22"]
g_idx, nir_idx, sw_idx = map(bands.index, ["green","nir","swir16"])
class_color = {"ice":(0,255,255),"water":(0,0,255),
               "cloud":(200,200,200),"land":(150,75,0),"nodata":(255,0,255)}

def preview(mask: np.ndarray)->Image.Image:
    hi = Image.fromarray((mask*255).astype(np.uint8),"L")
    return hi.resize((2048,2048),Image.NEAREST).resize((512,512),Image.BILINEAR)

def colour_overlay(rgb: Image.Image, ice, water, cloud, land, nodata)->Image.Image:
    base=rgb.convert("RGBA")
    overlay=np.zeros((*base.size[::-1],4),dtype=np.uint8)
    for m,lbl in [(ice,"ice"),(water,"water"),
                 (cloud,"cloud"),(land,"land"),(nodata,"nodata")]:
        mimg=Image.fromarray((m*255).astype(np.uint8)).resize(base.size,Image.NEAREST)
        overlay[np.array(mimg)>127]=(*class_color[lbl],120)
    return Image.alpha_composite(base,Image.fromarray(overlay,"RGBA")).convert("RGB")

# write CSV header
header = ["tile_id","timestamp",
          "ice_px","water_px","cloud_px","land_px","nodata_px","unknown_px",
          "ice_pct","water_pct","cloud_pct","land_pct","nodata_pct",
          "mean_ndsi_ice","mean_ndwi_water",
          "eo_cloud_cover","sun_elev","sun_azim","edge_gap"]
with open(CSV_FILE,"w",newline="") as f:
    csv.writer(f).writerow(header)

# ─── MAIN LOOP (per date) ─────────────────────────────────────
for date, tiles in sorted(by_date.items()):
    ts   = tiles[0].datetime.strftime("%Y%m%dT%H%M%S")
    meta = tiles[0].properties
    print(f"[INFO] {date}  tiles={len(tiles)}  ts={ts}")

    # include any touching neighbors
    union_geom = unary_union([box(*t.bbox) for t in tiles])
    for cand in items_raw:
        if cand.datetime.date()!=date or cand in tiles: continue
        if box(*cand.bbox).intersects(union_geom):
            print(f"   [DEBUG] adding neighbor {cand.id}")
            tiles.append(cand)
            union_geom = union_geom.union(box(*cand.bbox))

    # load & crop
    try:
        ds = load(tiles, geopolygon=SEARCH_AOI, groupby=None, chunks={})
    except Exception as e:
        print(f"   [WARN] load failed {date}: {e}")
        continue

    # build 10m RGB → 512px
    R,G,B = [ds[c].isel(time=0).values.astype(np.float32) for c in ("red","green","blue")]
    scale = 255.0 if R.max()<=1 else 255.0/10000.0
    rgb512 = Image.merge("RGB",[
        Image.fromarray(np.clip(ch*scale,0,255).astype(np.uint8),"L")
        for ch in (R,G,B)
    ]).resize((512,512),Image.BILINEAR)

    # downsample 40m cube
    cube = np.stack([ds[b].isel(time=0).values for b in bands]).astype(np.float32)
    if cube.max()>1.1: cube/=10000.0
    C,H,W = cube.shape
    H4,W4 = (H//4)*4,(W//4)*4
    small = F.avg_pool2d(torch.from_numpy(cube[None])[...,:H4,:W4],4,4).squeeze(0)
    s_np = small.cpu().numpy(); _,h4,w4 = small.shape

    # nodata & edge_gap
    nodata = (s_np.sum(0)<1e-6)
    nodata_px = int(nodata.sum()); nodata_pct = nodata_px/(h4*w4)
    edge_gap = int(nodata_pct>=NODATA_THR)
    if edge_gap: print(f"   [WARN] nodata {nodata_pct*100:.1f}% → edge_gap")

    # land mask
    land = np.array(Image.open(LANDMASK_FILE).convert("L").resize((w4,h4),Image.NEAREST))>127

    # cloud inference
    cloud, pcs, coords = np.zeros((h4,w4),bool),[],[]
    for y0 in range(0,h4,512):
        for x0 in range(0,w4,512):
            blk=small[:,y0:y0+512,x0:x0+512]; ph,pw=blk.shape[-2:]
            if (ph,pw)!=(512,512):
                pad=torch.zeros((C,512,512),device=device)
                pad[:,:ph,:pw]=blk; blk=pad
            pcs.append(blk); coords.append((y0,x0,ph,pw))
    if pcs:
        with autocast(device_type=device.type), torch.no_grad():
            pr=torch.softmax(model(torch.stack(pcs).to(device)),1)[:,1].cpu().numpy()
        for (y0,x0,ph,pw),p in zip(coords,pr):
            cloud[y0:y0+ph,x0:x0+pw]=p[:ph,:pw]>0.5

    # ND classes
    ndsi=(s_np[g_idx]-s_np[sw_idx])/(s_np[g_idx]+s_np[sw_idx]+1e-6)
    ndwi=(s_np[g_idx]-s_np[nir_idx])/(s_np[g_idx]+s_np[nir_idx]+1e-6)
    ice  =(ndsi>NDSI_THR)&~cloud&~land&~nodata
    water=(ndwi>NDWI_THR)&~ice&~cloud&~land&~nodata

    # stats
    total=h4*w4
    counts={"ice":int(ice.sum()),"water":int(water.sum()),
            "cloud":int(cloud.sum()),"land":int(land.sum()),
            "nodata":nodata_px}
    unknown=total-sum(counts.values())
    pct={k:v/total for k,v in counts.items()}
    mean_ndsi=float(np.nanmean(ndsi[ice])) if counts["ice"] else np.nan
    mean_ndwi=float(np.nanmean(ndwi[water]))if counts["water"]else np.nan

    # previews & overlay
    prev={n:preview(m) for n,m in [
        ("cloud",cloud),("ice",ice),("water",water),
        ("land",land),("nodata",nodata)
    ]}
    overlay=colour_overlay(rgb512,ice,water,cloud,land,nodata)
    overlay.save(f"{date}_{ts}_overlay512.png")

    # live panel
    fig,ax=plt.subplots(2,3,figsize=(13,8))
    ax[0,0].imshow(rgb512);            ax[0,0].set_title("RGB")
    ax[0,1].imshow(prev["cloud"],cmap="gray"); ax[0,1].set_title("Cloud")
    ax[0,2].imshow(prev["land"],cmap="gray");  ax[0,2].set_title("Land")
    ax[1,0].imshow(prev["ice"],cmap="gray");   ax[1,0].set_title("Sea‑Ice")
    ax[1,1].imshow(prev["water"],cmap="gray"); ax[1,1].set_title("Water")
    ax[1,2].imshow(overlay);                 ax[1,2].set_title("Overlay")
    for a in ax.flatten(): a.axis("off")
    plt.suptitle(f"{date}  {ts}",fontsize=11); plt.tight_layout(); plt.show()

    # save masks & append CSV
    prev["cloud"].save(f"{date}_{ts}_mask512.png")
    prev["ice"].save  (f"{date}_{ts}_ice512.png")
    prev["water"].save(f"{date}_{ts}_water512.png")
    prev["land"].save (f"{date}_{ts}_land512.png")

    with open(CSV_FILE,"a",newline="") as f:
        csv.writer(f).writerow([
            date, ts,
            counts["ice"],counts["water"],counts["cloud"],counts["land"],
            nodata_px,unknown,
            round(pct["ice"],4),round(pct["water"],4),
            round(pct["cloud"],4),round(pct["land"],4),round(nodata_pct,4),
            round(mean_ndsi,4) if not np.isnan(mean_ndsi) else "",
            round(mean_ndwi,4) if not np.isnan(mean_ndwi) else "",
            meta.get("eo:cloud_cover",""),
            meta.get("sat:solar_elevation",""),
            meta.get("sat:solar_azimuth",""),
            edge_gap
        ])

    # cleanup
    del ds,cube,small
    gc.collect()
    if torch.backends.mps.is_available(): torch.mps.empty_cache()
    else: torch.cuda.empty_cache()

print(f"\n[INFO] All done! Summary written to {CSV_FILE}")


# New approach as Script

In [ ]:
#!/usr/bin/env python3
"""
Fast CloudSEN12 UNetMobV2_V2 inference + sea-ice / water masks
‒ incremental & resumable – WITH live visual panel

    • one CSV row is flushed to disk right after each tile
    • pre-existing rows are skipped (non-destructive)
    • optional flag --no-viz disables the matplotlib window
"""

# ── imports ────────────────────────────────────────────────────
import os, sys, gc, csv, pathlib, argparse, datetime
import matplotlib.pyplot as plt
os.environ["AWS_NO_SIGN_REQUEST"] = "YES"

import numpy as np
if not hasattr(np,"round_"): np.round_ = np.round
import torch, torch.nn.functional as F
from torch.amp import autocast
from pystac_client import Client
from odc.stac import load
from PIL import Image
import segmentation_models_pytorch as smp

# ── CLI args ───────────────────────────────────────────────────
ap = argparse.ArgumentParser()
ap.add_argument("--start", type=int, default=2020, help="first year (inclusive)")
ap.add_argument("--end",   type=int, default=datetime.date.today().year,
                help="last year  (inclusive)")
ap.add_argument("--no-viz", action="store_true", help="skip matplotlib panels")
args = ap.parse_args()

# ── constants / user settings ─────────────────────────────────
CHECKPOINT_FILE = "UNetMobV2_V2.pt"
LANDMASK_FILE   = "landmask_template.png"
CSV_FILE        = "summary.csv"

SEARCH_AOI = {"type":"Polygon","coordinates":[[
    [-52.336121,70.788206],[-51.945564,70.788206],
    [-51.945564,70.628226],[-52.336121,70.628226],
    [-52.336121,70.788206]]]}

NDSI_THR, NDWI_THR = 0.42, 0.05
NODATA_THR         = 0.05  # 5 % nodata → edge_gap flag

bands   = ["coastal","blue","green","red","rededge1","rededge2","rededge3",
           "nir","nir08","nir09","cirrus","swir16","swir22"]
g_idx,nir_idx,sw_idx = map(bands.index, ["green","nir","swir16"])
class_color = {"ice":(0,255,255),"water":(0,0,255),
               "cloud":(200,200,200),"land":(150,75,0),"nodata":(255,0,255)}

# ── CSV init / dedup set ──────────────────────────────────────
header = ["tile_id","timestamp",
          "ice_px","water_px","cloud_px","land_px","nodata_px","unknown_px",
          "ice_pct","water_pct","cloud_pct","land_pct","nodata_pct",
          "mean_ndsi_ice","mean_ndwi_water",
          "eo_cloud_cover","sun_elev","sun_azim","edge_gap"]

existing = set()
csv_exists = pathlib.Path(CSV_FILE).is_file()
if csv_exists:
    with open(CSV_FILE) as fh:
        rdr = csv.reader(fh); next(rdr, None)
        existing = {(r[0],r[1]) for r in rdr}

csv_fh = open(CSV_FILE,"a",newline="")
csv_wr = csv.writer(csv_fh)
if not csv_exists:
    csv_wr.writerow(header)

# ── device & model ────────────────────────────────────────────
device = torch.device("mps") if torch.backends.mps.is_available() \
         else torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[INFO] device:", device)

model = smp.Unet("mobilenet_v2", encoder_weights=None,
                 in_channels=13, classes=4).to(device)
ckpt = torch.load(CHECKPOINT_FILE, map_location=device)
model.load_state_dict({k.removeprefix("module.").removeprefix("model."): v
                       for k,v in ckpt.get("model_state_dict", ckpt).items()},
                      strict=False)
model.eval(); print("[OK] model ready\n")

# ── helpers ───────────────────────────────────────────────────
def preview(mask: np.ndarray) -> Image.Image:
    hi = Image.fromarray((mask*255).astype(np.uint8),"L")
    return hi.resize((2048,2048),Image.NEAREST).resize((512,512),Image.BILINEAR)

def colour_overlay(rgb: Image.Image, ice, water, cloud, land, nodata):
    base = rgb.convert("RGBA")
    ov   = np.zeros((*base.size[::-1],4),np.uint8)
    for m,lbl in [(ice,"ice"),(water,"water"),(cloud,"cloud"),
                  (land,"land"),(nodata,"nodata")]:
        m_img = Image.fromarray((m*255).astype(np.uint8)).resize(base.size,Image.NEAREST)
        ov[np.array(m_img)>127] = (*class_color[lbl],120)
    return Image.alpha_composite(base,Image.fromarray(ov,"RGBA")).convert("RGB")

# ── STAC client ───────────────────────────────────────────────
client = Client.open("https://earth-search.aws.element84.com/v1")

# ── main year loop ────────────────────────────────────────────
for yr in range(args.start, args.end+1):
    drange = f"{yr}-01-01T00:00:00Z/{yr}-12-31T23:59:59Z"
    q = client.search(collections=["sentinel-2-l1c"],
                      intersects=SEARCH_AOI, datetime=drange)
    items = list(q.items())
    if not items:
        print(f"[WARN] {yr}: no tiles")
        continue
    items = {it.datetime.date(): it for it in items}.values()
    print(f"[INFO] {yr}: {len(items)} unique tiles")

    for it in items:
        ts = it.datetime.strftime("%Y%m%dT%H%M%S")
        key=(it.id,ts)
        if key in existing:
            print("   ↷",it.id,"already done")
            continue

        try:
            ds = load([it], geopolygon=SEARCH_AOI, groupby=None, chunks={})
        except Exception as e:
            print("   [ERR] load failed:", e)
            continue

        # ---------- 10-m RGB quicklook ------------------------
        R,G,B = [ds[c].isel(time=0).values.astype(np.float32)
                 for c in ("red","green","blue")]
        scale = 255.0 if R.max()<=1 else 255.0/10000.0
        rgb512 = Image.merge("RGB",[
            Image.fromarray(np.clip(c*scale,0,255).astype(np.uint8),"L")
            for c in (R,G,B)]).resize((512,512),Image.BILINEAR)

        # ---------- 40-m cube --------------------------------
        cube = np.stack([ds[b].isel(time=0).values for b in bands]).astype(np.float32)
        if cube.max()>1.1: cube/=10000.0
        C,H,W = cube.shape; H4,W4=(H//4)*4,(W//4)*4
        small = F.avg_pool2d(torch.from_numpy(cube[None])[...,:H4,:W4],4,4).squeeze(0)
        s_np  = small.numpy(); _,h4,w4 = small.shape

        nodata   = (s_np.sum(0)<1e-6)
        nodata_px= int(nodata.sum()); nodata_pct=nodata_px/(h4*w4)
        edge_gap = int(nodata_pct>=NODATA_THR)
        land = np.array(Image.open(LANDMASK_FILE).convert("L")
                        .resize((w4,h4),Image.NEAREST))>127

        # clouds
        cloud, pcs, coords = np.zeros((h4,w4),bool),[],[]
        for y0 in range(0,h4,512):
            for x0 in range(0,w4,512):
                blk = small[:,y0:y0+512,x0:x0+512]
                ph,pw = blk.shape[-2:]
                if (ph,pw)!=(512,512):
                    pad=torch.zeros((C,512,512)); pad[:,:ph,:pw]=blk; blk=pad
                pcs.append(blk); coords.append((y0,x0,ph,pw))
        if pcs:
            with autocast(device_type=device.type), torch.no_grad():
                pr = torch.softmax(model(torch.stack(pcs).to(device)),1)[:,1].cpu().numpy()
            for (y0,x0,ph,pw),p in zip(coords,pr):
                cloud[y0:y0+ph,x0:x0+pw]=p[:ph,:pw]>0.5

        ndsi=(s_np[g_idx]-s_np[sw_idx])/(s_np[g_idx]+s_np[sw_idx]+1e-6)
        ndwi=(s_np[g_idx]-s_np[nir_idx])/(s_np[g_idx]+s_np[nir_idx]+1e-6)
        ice  =(ndsi>NDSI_THR)&~cloud&~land&~nodata
        water=(ndwi>NDWI_THR)&~ice&~cloud&~land&~nodata

        total=h4*w4
        cnt={"ice":int(ice.sum()),"water":int(water.sum()),
             "cloud":int(cloud.sum()),"land":int(land.sum()),
             "nodata":nodata_px}
        unknown  = total-sum(cnt.values())
        pct      = {k:v/total for k,v in cnt.items()}
        mean_ndsi=float(np.nanmean(ndsi[ice])) if cnt["ice"] else np.nan
        mean_ndwi=float(np.nanmean(ndwi[water]))if cnt["water"]else np.nan

        overlay = colour_overlay(rgb512,ice,water,cloud,land,nodata)
        overlay.save(f"{it.id}_{ts}_overlay512.png")

        # ---------- live panel --------------------------------
        if not args.no_viz:
            prev = {n:preview(m) for n,m in
                    [("cloud",cloud),("ice",ice),("water",water),
                     ("land",land),("nodata",nodata)]}
            fig,ax=plt.subplots(2,3,figsize=(13,8))
            ax[0,0].imshow(rgb512);      ax[0,0].set_title("RGB");     ax[0,0].axis("off")
            ax[0,1].imshow(prev["cloud"],cmap="gray"); ax[0,1].set_title("Cloud"); ax[0,1].axis("off")
            ax[0,2].imshow(prev["land"],cmap="gray");  ax[0,2].set_title("Land");  ax[0,2].axis("off")
            ax[1,0].imshow(prev["ice"],cmap="gray");   ax[1,0].set_title("Sea-ice");ax[1,0].axis("off")
            ax[1,1].imshow(prev["water"],cmap="gray"); ax[1,1].set_title("Water"); ax[1,1].axis("off")
            ax[1,2].imshow(overlay);     ax[1,2].set_title("Overlay"); ax[1,2].axis("off")
            plt.suptitle(f"{it.id}  {ts}",fontsize=11); plt.tight_layout(); plt.show(block=False)
            plt.pause(0.001)
            plt.close(fig)

        # ---------- CSV flush ---------------------------------
        csv_wr.writerow([
            it.id, ts,
            cnt["ice"],cnt["water"],cnt["cloud"],cnt["land"],
            nodata_px,unknown,
            round(pct["ice"],4),round(pct["water"],4),
            round(pct["cloud"],4),round(pct["land"],4),round(nodata_pct,4),
            round(mean_ndsi,4) if not np.isnan(mean_ndsi) else "",
            round(mean_ndwi,4) if not np.isnan(mean_ndwi) else "",
            it.properties.get("eo:cloud_cover",""),
            it.properties.get("sat:solar_elevation",""),
            it.properties.get("sat:solar_azimuth",""),
            edge_gap
        ])
        csv_fh.flush()
        existing.add(key)

        # ---------- cleanup -----------------------------------
        del ds,cube,small; gc.collect()
        if torch.backends.mps.is_available(): torch.mps.empty_cache()
        elif torch.cuda.is_available(): torch.cuda.empty_cache()

csv_fh.close()
print("\n[✓] Done – rows appended to", CSV_FILE)
